# SAM2 Encoder-Adapter Experiment — All Datasets
Standalone notebook: environment setup, data loading (VegAnn, ReLeaf, WeedsGalore, EWI1S), AgriCLIP box generation, and the decoder-only vs. decoder+encoder-adapter comparison across all four datasets.


## 1. Imports and environment

In [1]:
import os
import sys
import json
import random
import math
import copy
import time
import csv
import warnings

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import cv2

from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    DataLoader,
)

from datasets import load_dataset

from transformers import (
    Sam2Model,
    Sam2Processor,
)

from IPython.display import display

warnings.filterwarnings("ignore")

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)

Python: 3.10.12 (main, Jun 22 2026, 18:55:27) [GCC 11.4.0]
PyTorch: 2.12.0+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 4090
CUDA: 13.0


In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE = device

MODEL_NAME = "facebook/sam2.1-hiera-tiny"

BATCH_SIZE = 1
NUM_WORKERS = 0

# Prompt perturbation used only during training
BOX_JITTER_FRACTION = 0.05

# Evaluation / training mask threshold
MASK_THRESHOLD = 0.5

print("Device:", device)
print("Model:", MODEL_NAME)

Device: cuda
Model: facebook/sam2.1-hiera-tiny


## 2. Dataset paths
**Edit `PROJECT_ROOT` if your data lives somewhere else.**

In [3]:
PROJECT_ROOT = Path("/home/zahranadeem/thesis")

DOMAIN_FT_ROOT = PROJECT_ROOT / "sam2_domain_finetuning"
DOMAIN_FT_ROOT.mkdir(parents=True, exist_ok=True)

# ============================================================
# Existing VegAnn SAM2 checkpoint
# ============================================================

VEGANN_SAM2_CHECKPOINT = (
    PROJECT_ROOT / "sam2_vegann_checkpoints" / "best_sam2_vegann_mask_decoder.pt"
)

# ============================================================
# ReLeaf
# ============================================================

RELEAF_ROOT = PROJECT_ROOT / "CropAndWeedAndLeaf"
RELEAF_REPO = PROJECT_ROOT / "releaf"
RELEAF_IMAGES_DIR = RELEAF_ROOT / "images"
RELEAF_LABELS_DIR = RELEAF_ROOT / "labels"
RELEAF_TEST_DIR = RELEAF_REPO / "data" / "CWL" / "test"

# ============================================================
# WeedsGalore
# ============================================================

WEEDSGALORE_ROOT = PROJECT_ROOT / "WeedsGalore_dataset"
WEEDS_IMAGE_DIR = WEEDSGALORE_ROOT / "data"
WEEDS_MASK_DIR = WEEDSGALORE_ROOT / "fields" / "ground_truth_semantic"

# ============================================================
# EWI1S
# ============================================================

EWIS1_ROOT = PROJECT_ROOT / "EWIS1"


def find_nested_dataset_dir(root, dirname, extension=".png"):
    """Search recursively because extracted archives can nest
    images/images or masks/masks directories."""
    candidates = []
    for p in root.rglob(dirname):
        if not p.is_dir():
            continue
        files = [x for x in p.iterdir()
                 if x.is_file() and x.suffix.lower() == extension.lower()]
        if len(files) > 0:
            candidates.append((len(files), p))
    if not candidates:
        return None
    candidates.sort(reverse=True, key=lambda x: x[0])
    return candidates[0][1]


EWIS1_IMAGE_DIR = find_nested_dataset_dir(EWIS1_ROOT, "images", ".png")
EWIS1_MASK_DIR = find_nested_dataset_dir(EWIS1_ROOT, "masks", ".png")

print("VegAnn SAM2 checkpoint:", VEGANN_SAM2_CHECKPOINT)
print("\nReLeaf images:", RELEAF_IMAGES_DIR)
print("ReLeaf labels:", RELEAF_LABELS_DIR)
print("ReLeaf test:", RELEAF_TEST_DIR)
print("\nWeedsGalore images:", WEEDS_IMAGE_DIR)
print("WeedsGalore masks:", WEEDS_MASK_DIR)
print("\nEWI1S images:", EWIS1_IMAGE_DIR)
print("EWI1S masks:", EWIS1_MASK_DIR)

VegAnn SAM2 checkpoint: /home/zahranadeem/thesis/sam2_vegann_checkpoints/best_sam2_vegann_mask_decoder.pt

ReLeaf images: /home/zahranadeem/thesis/CropAndWeedAndLeaf/images
ReLeaf labels: /home/zahranadeem/thesis/CropAndWeedAndLeaf/labels
ReLeaf test: /home/zahranadeem/thesis/releaf/data/CWL/test

WeedsGalore images: /home/zahranadeem/thesis/WeedsGalore_dataset/data
WeedsGalore masks: /home/zahranadeem/thesis/WeedsGalore_dataset/fields/ground_truth_semantic

EWI1S images: /home/zahranadeem/thesis/EWIS1/EWIS1 Maize, Sorghum & Weed Dataset/images/images
EWI1S masks: /home/zahranadeem/thesis/EWIS1/EWIS1 Maize, Sorghum & Weed Dataset/masks/masks


In [4]:
paths_to_check = {
    "VegAnn SAM2 checkpoint": VEGANN_SAM2_CHECKPOINT,
    "ReLeaf images": RELEAF_IMAGES_DIR,
    "ReLeaf labels": RELEAF_LABELS_DIR,
    "WeedsGalore images": WEEDS_IMAGE_DIR,
    "WeedsGalore masks": WEEDS_MASK_DIR,
    "EWI1S images": EWIS1_IMAGE_DIR,
    "EWI1S masks": EWIS1_MASK_DIR,
}

missing_paths = []
for name, path in paths_to_check.items():
    ok = path is not None and Path(path).exists()
    print(f"{name:30s}: {'OK' if ok else 'MISSING'}")
    if not ok:
        missing_paths.append(name)

if missing_paths:
    raise FileNotFoundError(
        "The following paths were not found on disk: "
        f"{missing_paths}. Fix PROJECT_ROOT / the dataset subpaths above "
        "before continuing — everything below depends on this data."
    )

VegAnn SAM2 checkpoint        : OK
ReLeaf images                 : OK
ReLeaf labels                 : OK
WeedsGalore images            : OK
WeedsGalore masks             : OK
EWI1S images                  : OK
EWI1S masks                   : OK


In [5]:
def count_files(directory, extensions):
    if directory is None or not Path(directory).exists():
        return 0
    return sum(1 for p in Path(directory).rglob("*")
               if p.is_file() and p.suffix.lower() in extensions)


IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}
MASK_EXTENSIONS = IMAGE_EXTENSIONS

print("ReLeaf images:", count_files(RELEAF_IMAGES_DIR, IMAGE_EXTENSIONS))
print("ReLeaf labels:", count_files(RELEAF_LABELS_DIR, {".txt"}))
print("WeedsGalore images:", count_files(WEEDS_IMAGE_DIR, IMAGE_EXTENSIONS))
print("WeedsGalore masks:", count_files(WEEDS_MASK_DIR, MASK_EXTENSIONS))
print("EWI1S images:", count_files(EWIS1_IMAGE_DIR, IMAGE_EXTENSIONS))
print("EWI1S masks:", count_files(EWIS1_MASK_DIR, MASK_EXTENSIONS))

ReLeaf images: 345
ReLeaf labels: 345
WeedsGalore images: 156
WeedsGalore masks: 156
EWI1S images: 88
EWI1S masks: 88


## 3. Load VegAnn and split

In [6]:
vegann = load_dataset("simonMadec/VegAnn")

print(vegann)
print("Columns:", vegann["train"].column_names)
print("Total images:", len(vegann["train"]))

vegann_split = vegann["train"].train_test_split(test_size=0.20, seed=SEED)

vegann_train_hf = vegann_split["train"]
vegann_val_hf = vegann_split["test"]

print("VegAnn training:", len(vegann_train_hf))
print("VegAnn validation:", len(vegann_val_hf))

DatasetDict({
    train: Dataset({
        features: ['image', 'mask', 'System', 'Orientation', 'latitude', 'longitude', 'date', 'LocAcc', 'Species', 'Owner', 'Dataset-Name', 'TVT-split1', 'TVT-split2', 'TVT-split3', 'TVT-split4', 'TVT-split5'],
        num_rows: 3775
    })
})
Columns: ['image', 'mask', 'System', 'Orientation', 'latitude', 'longitude', 'date', 'LocAcc', 'Species', 'Owner', 'Dataset-Name', 'TVT-split1', 'TVT-split2', 'TVT-split3', 'TVT-split4', 'TVT-split5']
Total images: 3775
VegAnn training: 3020
VegAnn validation: 755


## 4. Shared mask / box utilities

In [7]:
def binary_mask_from_array(mask):
    """Convert any mask-like object into a uint8 binary mask.
    Any non-zero pixel is foreground."""
    if isinstance(mask, Image.Image):
        mask = np.array(mask)
    elif torch.is_tensor(mask):
        mask = mask.detach().cpu().numpy()

    mask = np.asarray(mask)
    while mask.ndim > 2:
        mask = mask[..., 0]

    return (mask > 0).astype(np.uint8)


def mask_to_box(mask):
    """Compute [x1, y1, x2, y2] from a binary mask. None for empty masks."""
    mask = binary_mask_from_array(mask)
    ys, xs = np.where(mask > 0)
    if len(xs) == 0:
        return None
    x1, y1 = int(xs.min()), int(ys.min())
    x2, y2 = int(xs.max()), int(ys.max())
    return np.array([x1, y1, x2, y2], dtype=np.int32)


def jitter_box(box, image_width, image_height, fraction=BOX_JITTER_FRACTION):
    """Randomly shift/expand/shrink a GT box slightly. Training only."""
    x1, y1, x2, y2 = [int(v) for v in box]

    box_width = max(1, x2 - x1)
    box_height = max(1, y2 - y1)

    max_dx = int(box_width * fraction)
    max_dy = int(box_height * fraction)

    dx1 = random.randint(-max_dx, max_dx)
    dy1 = random.randint(-max_dy, max_dy)
    dx2 = random.randint(-max_dx, max_dx)
    dy2 = random.randint(-max_dy, max_dy)

    x1 = max(0, min(x1 + dx1, image_width - 1))
    y1 = max(0, min(y1 + dy1, image_height - 1))
    x2 = max(x1 + 1, min(x2 + dx2, image_width - 1))
    y2 = max(y1 + 1, min(y2 + dy2, image_height - 1))

    return np.array([x1, y1, x2, y2], dtype=np.int32)

## 5. ReLeaf loader

In [8]:
def find_matching_mask_or_label(directory, image_path):
    """Find a file with the same stem (any of the given suffixes)."""
    for suffix in [".txt", ".png", ".jpg", ".jpeg", ".tif", ".tiff"]:
        candidate = directory / f"{image_path.stem}{suffix}"
        if candidate.exists():
            return candidate
    return None


def load_releaf_image_mask(image_path):
    """ReLeaf: YOLO segmentation polygons -> binary vegetation mask."""
    image = Image.open(image_path).convert("RGB")
    width, height = image.size
    mask = np.zeros((height, width), dtype=np.uint8)

    label_path = RELEAF_LABELS_DIR / f"{image_path.stem}.txt"
    if not label_path.exists():
        return image, mask

    with open(label_path, "r") as f:
        lines = f.readlines()

    for line in lines:
        values = line.strip().split()
        if len(values) < 7:
            continue
        coords = [float(v) for v in values[1:]]  # skip class id
        if len(coords) < 6:
            continue

        points = []
        for i in range(0, len(coords), 2):
            x = int(coords[i] * width)
            y = int(coords[i + 1] * height)
            points.append([x, y])
        points = np.asarray(points, dtype=np.int32)
        cv2.fillPoly(mask, [points], 1)

    return image, mask

In [9]:
releaf_pairs = []

releaf_image_files = sorted(
    p for p in RELEAF_IMAGES_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
)

for image_path in releaf_image_files:
    label_path = RELEAF_LABELS_DIR / f"{image_path.stem}.txt"
    if label_path.exists():
        releaf_pairs.append({
            "image_path": image_path,
            "label_path": label_path,
            "dataset": "ReLeaf",
        })

print("ReLeaf image-label pairs:", len(releaf_pairs))
print(releaf_pairs[:3])

# Note: we deliberately do NOT exclude the official ReLeaf test-split
# images here — we build our own 80/10/10 split over all matched
# pairs below (see split_pairs), so nothing needs excluding.

ReLeaf image-label pairs: 345
[{'image_path': PosixPath('/home/zahranadeem/thesis/CropAndWeedAndLeaf/images/ave-0041-0002-002.jpg'), 'label_path': PosixPath('/home/zahranadeem/thesis/CropAndWeedAndLeaf/labels/ave-0041-0002-002.txt'), 'dataset': 'ReLeaf'}, {'image_path': PosixPath('/home/zahranadeem/thesis/CropAndWeedAndLeaf/images/ave-0045-0011-012.jpg'), 'label_path': PosixPath('/home/zahranadeem/thesis/CropAndWeedAndLeaf/labels/ave-0045-0011-012.txt'), 'dataset': 'ReLeaf'}, {'image_path': PosixPath('/home/zahranadeem/thesis/CropAndWeedAndLeaf/images/ave-0048-0002-026.jpg'), 'label_path': PosixPath('/home/zahranadeem/thesis/CropAndWeedAndLeaf/labels/ave-0048-0002-026.txt'), 'dataset': 'ReLeaf'}]


## 6. WeedsGalore pair builder

In [10]:
def find_mask_for_stem(mask_dir, stem):
    """Search for a matching mask regardless of extension."""
    matches = [p for p in mask_dir.glob(f"{stem}.*")
               if p.suffix.lower() in MASK_EXTENSIONS]
    return matches[0] if matches else None


weedsgalore_pairs = []

weed_image_files = sorted(
    p for p in WEEDS_IMAGE_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
)

for image_path in weed_image_files:
    mask_path = find_mask_for_stem(WEEDS_MASK_DIR, image_path.stem)
    if mask_path is not None:
        weedsgalore_pairs.append({
            "image_path": image_path,
            "mask_path": mask_path,
            "dataset": "WeedsGalore",
        })

print("WeedsGalore matched pairs:", len(weedsgalore_pairs))
print(weedsgalore_pairs[:3])

WeedsGalore matched pairs: 156
[{'image_path': PosixPath('/home/zahranadeem/thesis/WeedsGalore_dataset/data/2023-05-25_0109.png'), 'mask_path': PosixPath('/home/zahranadeem/thesis/WeedsGalore_dataset/fields/ground_truth_semantic/2023-05-25_0109.png'), 'dataset': 'WeedsGalore'}, {'image_path': PosixPath('/home/zahranadeem/thesis/WeedsGalore_dataset/data/2023-05-25_0114.png'), 'mask_path': PosixPath('/home/zahranadeem/thesis/WeedsGalore_dataset/fields/ground_truth_semantic/2023-05-25_0114.png'), 'dataset': 'WeedsGalore'}, {'image_path': PosixPath('/home/zahranadeem/thesis/WeedsGalore_dataset/data/2023-05-25_0119.png'), 'mask_path': PosixPath('/home/zahranadeem/thesis/WeedsGalore_dataset/fields/ground_truth_semantic/2023-05-25_0119.png'), 'dataset': 'WeedsGalore'}]


In [11]:
def load_weedsgalore_image_mask(item):
    image = Image.open(item["image_path"]).convert("RGB")
    mask = Image.open(item["mask_path"])
    mask = binary_mask_from_array(mask)
    return image, mask


# Inspect a sample
_sample_image, _sample_mask = load_weedsgalore_image_mask(weedsgalore_pairs[0])
print("Image size:", _sample_image.size)
print("Mask shape:", _sample_mask.shape)
print("Mask values:", np.unique(_sample_mask))
print("Foreground pixels:", int(_sample_mask.sum()))

Image size: (600, 600)
Mask shape: (600, 600)
Mask values: [0 1]
Foreground pixels: 975


## 7. EWI1S pair builder

In [12]:
ewis1_pairs = []

ewis1_image_files = sorted(
    p for p in EWIS1_IMAGE_DIR.iterdir()
    if p.is_file() and p.suffix.lower() == ".png"
)

for image_path in ewis1_image_files:
    mask_path = EWIS1_MASK_DIR / image_path.name
    if mask_path.exists():
        ewis1_pairs.append({
            "image_path": image_path,
            "mask_path": mask_path,
            "dataset": "EWI1S",
        })

print("EWI1S matched pairs:", len(ewis1_pairs))

EWI1S matched pairs: 88


In [13]:
def load_ewis1_image_mask(item):
    image = Image.open(item["image_path"]).convert("RGB")
    mask_rgba = np.array(Image.open(item["mask_path"]).convert("RGBA"))

    rgb = mask_rgba[:, :, :3]
    alpha = mask_rgba[:, :, 3]

    # Official EWI1S weed color
    weed_pixels = (
        (rgb[:, :, 0] == 255) & (rgb[:, :, 1] == 127) & (rgb[:, :, 2] == 14)
        & (alpha > 0)
    )
    weed_mask = weed_pixels.astype(np.uint8)
    return image, weed_mask


# Verify sample
_sample_image, _sample_mask = load_ewis1_image_mask(ewis1_pairs[0])
print("Image size:", _sample_image.size)
print("Mask shape:", _sample_mask.shape)
print("Mask values:", np.unique(_sample_mask))
print("Weed pixels:", int(_sample_mask.sum()))

Image size: (5464, 3640)
Mask shape: (3640, 5464)
Mask values: [0 1]
Weed pixels: 48261


## 8. Train/val/test splits for all four datasets
`split_pairs` gives a deterministic 80/10/10 split. VegAnn already has its own HuggingFace train/val split from step 3, so only the other three datasets go through this.

In [14]:
def split_pairs(pairs, seed=42, train_fraction=0.80, val_fraction=0.10):
    """Deterministic 80/10/10 split."""
    pairs = list(pairs)
    rng = random.Random(seed)
    rng.shuffle(pairs)

    n = len(pairs)
    n_train = int(train_fraction * n)
    n_val = int(val_fraction * n)

    train = pairs[:n_train]
    val = pairs[n_train:n_train + n_val]
    test = pairs[n_train + n_val:]
    return train, val, test


releaf_train_pairs, releaf_val_pairs, releaf_test_pairs = split_pairs(
    releaf_pairs, seed=SEED
)

weedsgalore_train_pairs, weedsgalore_val_pairs, weedsgalore_test_pairs = split_pairs(
    weedsgalore_pairs, seed=SEED
)

ewis1_train_pairs, ewis1_val_pairs, ewis1_test_pairs = split_pairs(
    ewis1_pairs, seed=SEED
)

print("ReLeaf      | train:", len(releaf_train_pairs),
      "val:", len(releaf_val_pairs), "test:", len(releaf_test_pairs))
print("WeedsGalore | train:", len(weedsgalore_train_pairs),
      "val:", len(weedsgalore_val_pairs), "test:", len(weedsgalore_test_pairs))
print("EWI1S       | train:", len(ewis1_train_pairs),
      "val:", len(ewis1_val_pairs), "test:", len(ewis1_test_pairs))

ReLeaf      | train: 276 val: 34 test: 35
WeedsGalore | train: 124 val: 15 test: 17
EWI1S       | train: 70 val: 8 test: 10


In [15]:
def filter_nonempty_pairs(pairs, loader_function):
    """Drop any (image, mask) pair whose mask is entirely empty —
    UnifiedSegmentationDataset raises on an empty mask, so these
    must be removed from every pool used for training/validation."""
    valid, empty = [], []
    for item in tqdm(pairs, desc="Checking masks"):
        _, mask = loader_function(item)
        (valid if mask.sum() > 0 else empty).append(item)
    return valid, empty


releaf_train_pairs, _dropped = filter_nonempty_pairs(
    releaf_train_pairs, lambda item: load_releaf_image_mask(item["image_path"])
)
print("ReLeaf train — dropped empty-mask pairs:", len(_dropped))

releaf_val_pairs, _dropped = filter_nonempty_pairs(
    releaf_val_pairs, lambda item: load_releaf_image_mask(item["image_path"])
)
print("ReLeaf val   — dropped empty-mask pairs:", len(_dropped))

weedsgalore_train_pairs, _dropped = filter_nonempty_pairs(
    weedsgalore_train_pairs, load_weedsgalore_image_mask
)
print("WeedsGalore train — dropped empty-mask pairs:", len(_dropped))

weedsgalore_val_pairs, _dropped = filter_nonempty_pairs(
    weedsgalore_val_pairs, load_weedsgalore_image_mask
)
print("WeedsGalore val   — dropped empty-mask pairs:", len(_dropped))

ewis1_train_pairs, _dropped = filter_nonempty_pairs(
    ewis1_train_pairs, load_ewis1_image_mask
)
print("EWI1S train — dropped empty-mask pairs:", len(_dropped))

ewis1_val_pairs, _dropped = filter_nonempty_pairs(
    ewis1_val_pairs, load_ewis1_image_mask
)
print("EWI1S val   — dropped empty-mask pairs:", len(_dropped))

print("\nFinal sizes:")
print("ReLeaf      | train:", len(releaf_train_pairs), "val:", len(releaf_val_pairs))
print("WeedsGalore | train:", len(weedsgalore_train_pairs), "val:", len(weedsgalore_val_pairs))
print("EWI1S       | train:", len(ewis1_train_pairs), "val:", len(ewis1_val_pairs))

Checking masks:   0%|          | 0/276 [00:00<?, ?it/s]

ReLeaf train — dropped empty-mask pairs: 0


Checking masks:   0%|          | 0/34 [00:00<?, ?it/s]

ReLeaf val   — dropped empty-mask pairs: 0


Checking masks:   0%|          | 0/124 [00:00<?, ?it/s]

WeedsGalore train — dropped empty-mask pairs: 0


... (2 outputs left out)

Checking masks:   0%|          | 0/70 [00:00<?, ?it/s]

EWI1S train — dropped empty-mask pairs: 0


Checking masks:   0%|          | 0/8 [00:00<?, ?it/s]

EWI1S val   — dropped empty-mask pairs: 0

Final sizes:
ReLeaf      | train: 276 val: 34
WeedsGalore | train: 124 val: 15
EWI1S       | train: 70 val: 8


## 9. Unified dataset class

In [16]:
class UnifiedSegmentationDataset(Dataset):

    def __init__(self, records, dataset_name, jitter=False):
        self.records = records
        self.dataset_name = dataset_name
        self.jitter = jitter

    def __len__(self):
        return len(self.records)

    def _load_item(self, record):
        if self.dataset_name == "VegAnn":
            hf_dataset = record["hf_dataset"]
            sample = hf_dataset[record["index"]]
            image = sample["image"]
            mask = sample["mask"]
            if not isinstance(image, Image.Image):
                image = Image.fromarray(np.array(image))
            mask = binary_mask_from_array(mask)
            return image.convert("RGB"), mask

        elif self.dataset_name == "ReLeaf":
            return load_releaf_image_mask(record["image_path"])

        elif self.dataset_name == "WeedsGalore":
            return load_weedsgalore_image_mask(record)

        elif self.dataset_name == "EWI1S":
            return load_ewis1_image_mask(record)

        else:
            raise ValueError(f"Unknown dataset: {self.dataset_name}")

    def __getitem__(self, index):
        record = self.records[index]
        image, mask = self._load_item(record)

        box = mask_to_box(mask)
        if box is None:
            raise RuntimeError(
                f"Empty mask encountered in {self.dataset_name} at index {index}. "
                "This pool should have been filtered with filter_nonempty_pairs."
            )

        width, height = image.size
        box_for_model = jitter_box(box, width, height) if self.jitter else box.copy()

        return {
            "image": image,
            "mask": mask,
            "box": box_for_model,
            "gt_box": box,
            "index": index,
            "dataset": self.dataset_name,
        }

In [17]:
# Build VegAnn records, excluding empty masks

vegann_train_records = []
for i in range(len(vegann_train_hf)):
    mask = binary_mask_from_array(vegann_train_hf[i]["mask"])
    if mask.sum() > 0:
        vegann_train_records.append({"hf_dataset": vegann_train_hf, "index": i})

vegann_val_records = []
for i in range(len(vegann_val_hf)):
    mask = binary_mask_from_array(vegann_val_hf[i]["mask"])
    if mask.sum() > 0:
        vegann_val_records.append({"hf_dataset": vegann_val_hf, "index": i})

print("VegAnn usable train records:", len(vegann_train_records))
print("VegAnn usable val records:", len(vegann_val_records))

VegAnn usable train records: 2966
VegAnn usable val records: 746


In [18]:
datasets_train = {}
datasets_val = {}

datasets_train["VegAnn"] = UnifiedSegmentationDataset(vegann_train_records, "VegAnn", jitter=True)
datasets_val["VegAnn"] = UnifiedSegmentationDataset(vegann_val_records, "VegAnn", jitter=False)

datasets_train["ReLeaf"] = UnifiedSegmentationDataset(releaf_train_pairs, "ReLeaf", jitter=True)
datasets_val["ReLeaf"] = UnifiedSegmentationDataset(releaf_val_pairs, "ReLeaf", jitter=False)

datasets_train["WeedsGalore"] = UnifiedSegmentationDataset(weedsgalore_train_pairs, "WeedsGalore", jitter=True)
datasets_val["WeedsGalore"] = UnifiedSegmentationDataset(weedsgalore_val_pairs, "WeedsGalore", jitter=False)

datasets_train["EWI1S"] = UnifiedSegmentationDataset(ewis1_train_pairs, "EWI1S", jitter=True)
datasets_val["EWI1S"] = UnifiedSegmentationDataset(ewis1_val_pairs, "EWI1S", jitter=False)

for name in datasets_train:
    print(f"{name:15s} | train={len(datasets_train[name]):5d} | val={len(datasets_val[name]):5d}")

VegAnn          | train= 2966 | val=  746
ReLeaf          | train=  276 | val=   34
WeedsGalore     | train=  124 | val=   15
EWI1S           | train=   70 | val=    8


In [19]:
# Sanity check every dataset's first item before touching SAM2
for name, dataset in datasets_train.items():
    item = dataset[0]
    print("\n" + "=" * 60)
    print(name)
    print("Image:", item["image"].size)
    print("Mask:", item["mask"].shape, np.unique(item["mask"]))
    print("GT box:", item["gt_box"])
    print("Training box:", item["box"])


VegAnn
Image: (512, 512)
Mask: (512, 512) [0 1]
GT box: [ 45   0 454 386]
Training box: [ 65   0 435 384]

ReLeaf
Image: (272, 199)
Mask: (199, 272) [0 1]
GT box: [  0   0 271 198]
Training box: [  0   0 262 192]

WeedsGalore
Image: (600, 600)
Mask: (600, 600) [0 1]
GT box: [  0   0 599 599]
Training box: [ 14  18 599 599]

EWI1S
Image: (5464, 3640)
Mask: (3640, 5464) [0 1]
GT box: [   0    0 5463 3639]
Training box: [   0  121 5463 3474]


## 10. SAM2 processor, collate function, batch prep

In [20]:
def sam2_collate_fn(batch):
    return {
        "images": [item["image"] for item in batch],
        "masks": [item["mask"] for item in batch],
        "boxes": [item["box"] for item in batch],
        "gt_boxes": [item["gt_box"] for item in batch],
        "indices": [item["index"] for item in batch],
        "datasets": [item["dataset"] for item in batch],
    }


processor = Sam2Processor.from_pretrained(MODEL_NAME)
print("SAM2 processor loaded.")

SAM2 processor loaded.


In [21]:
def prepare_sam2_batch(batch):
    images = batch["images"]
    masks = batch["masks"]
    boxes = batch["boxes"]

    input_boxes = [[box.tolist()] for box in boxes]

    inputs = processor(
        images=images,
        segmentation_maps=masks,
        input_boxes=input_boxes,
        return_tensors="pt",
    )

    inputs = {
        key: (value.to(device) if torch.is_tensor(value) else value)
        for key, value in inputs.items()
    }
    return inputs

## 11. Losses and metrics

In [22]:
bce_loss_fn = nn.BCEWithLogitsLoss()


def dice_loss(logits, targets, smooth=1.0):
    probabilities = torch.sigmoid(logits).flatten(1)
    targets = targets.flatten(1)
    intersection = (probabilities * targets).sum(dim=1)
    dice = (2.0 * intersection + smooth) / (
        probabilities.sum(dim=1) + targets.sum(dim=1) + smooth
    )
    return 1.0 - dice.mean()


def combined_segmentation_loss(logits, targets):
    bce = bce_loss_fn(logits, targets)
    dice = dice_loss(logits, targets)
    total = 0.5 * bce + 0.5 * dice
    return total, bce, dice


def extract_single_mask_logits(pred_masks):
    """Convert SAM2 mask output to [B, H, W]."""
    logits = pred_masks
    while logits.ndim > 3:
        logits = logits[:, 0]
    return logits


def compute_batch_metrics(logits, targets, threshold=0.5, epsilon=1e-7):
    probabilities = torch.sigmoid(logits)
    predictions = (probabilities >= threshold).float()

    predictions = predictions.flatten(1)
    targets = targets.flatten(1)

    intersection = (predictions * targets).sum(dim=1)
    prediction_sum = predictions.sum(dim=1)
    target_sum = targets.sum(dim=1)
    union = prediction_sum + target_sum - intersection

    per_image_iou = (intersection + epsilon) / (union + epsilon)
    per_image_dice = (2 * intersection + epsilon) / (prediction_sum + target_sum + epsilon)

    return {
        "per_image_iou_sum": per_image_iou.sum().item(),
        "per_image_dice_sum": per_image_dice.sum().item(),
        "intersection": intersection.sum().item(),
        "union": union.sum().item(),
        "prediction_sum": prediction_sum.sum().item(),
        "target_sum": target_sum.sum().item(),
        "num_images": targets.shape[0],
    }

In [23]:
@torch.no_grad()
def validate(model, loader, threshold=0.5):
    """Validation loop for a plain (no-adapter) SAM2 decoder-only model,
    using GT boxes via prepare_sam2_batch. Used for the decoder-only side
    of the GT-box comparison."""
    model.eval()
    running_loss = 0.0
    totals = {"per_image_iou_sum": 0.0, "per_image_dice_sum": 0.0,
              "intersection": 0.0, "union": 0.0,
              "prediction_sum": 0.0, "target_sum": 0.0, "num_images": 0}

    for batch in loader:
        inputs = prepare_sam2_batch(batch)
        outputs = model(
            pixel_values=inputs["pixel_values"],
            input_boxes=inputs["input_boxes"],
            multimask_output=False,
        )
        logits = extract_single_mask_logits(outputs.pred_masks)
        targets = inputs["labels"].float()
        while targets.ndim > 3:
            targets = targets[:, 0]
        targets = F.interpolate(
            targets.unsqueeze(1), size=logits.shape[-2:], mode="nearest"
        ).squeeze(1)
        targets = (targets > 0).float()

        loss, _, _ = combined_segmentation_loss(logits, targets)
        running_loss += loss.item()

        batch_metrics = compute_batch_metrics(logits, targets, threshold=threshold)
        for key in totals:
            if key in batch_metrics:
                totals[key] += batch_metrics[key]

    n_images = totals["num_images"]
    if n_images == 0:
        raise RuntimeError("No validation images were evaluated.")

    return {
        "loss": running_loss / len(loader),
        "mean_iou": totals["per_image_iou_sum"] / n_images,
        "mean_dice": totals["per_image_dice_sum"] / n_images,
        "global_iou": totals["intersection"] / (totals["union"] + 1e-7),
        "global_dice": (2 * totals["intersection"]
                         / (totals["prediction_sum"] + totals["target_sum"] + 1e-7)),
    }

print("validate() ready.")

validate() ready.


## 12. Loading a decoder checkpoint into a fresh SAM2 model

In [24]:
def create_sam2_from_decoder_checkpoint(checkpoint_path):
    model = Sam2Model.from_pretrained(MODEL_NAME)

    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)

    if isinstance(checkpoint, dict) and "mask_decoder_state_dict" in checkpoint:
        decoder_state = checkpoint["mask_decoder_state_dict"]
    elif isinstance(checkpoint, dict):
        decoder_state = checkpoint
    else:
        raise TypeError(f"Unsupported checkpoint structure: {type(checkpoint)}")

    model.mask_decoder.load_state_dict(decoder_state, strict=True)
    model = model.to(DEVICE)
    return model, checkpoint

print("Decoder checkpoint loader ready.")

Decoder checkpoint loader ready.


In [25]:
if not VEGANN_SAM2_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Existing VegAnn SAM2 checkpoint not found:\n{VEGANN_SAM2_CHECKPOINT}"
    )

_baseline_model, _baseline_ckpt = create_sam2_from_decoder_checkpoint(VEGANN_SAM2_CHECKPOINT)
print("VegAnn checkpoint loaded OK.")
print("Checkpoint epoch:", _baseline_ckpt.get("epoch", "N/A") if isinstance(_baseline_ckpt, dict) else "N/A")
print("Checkpoint best IoU:", _baseline_ckpt.get("best_mean_iou", "N/A") if isinstance(_baseline_ckpt, dict) else "N/A")

del _baseline_model, _baseline_ckpt
torch.cuda.empty_cache()

[transformers] You are using a model of type `sam2_video` to instantiate a model of type `sam2`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Loading weights:   0%|          | 0/309 [00:00<?, ?it/s]

VegAnn checkpoint loaded OK.
Checkpoint epoch: 8
Checkpoint best IoU: 0.8724805513351916


## 13. Lightweight encoder feature adapter

In [ ]:
# ============================================================
# Multi-scale SAM2 Encoder Feature Adapter as SAM2 Hiera-Tiny produces 3 
# ============================================================

import torch
import torch.nn as nn


class SAM2MultiScaleFeatureAdapter(nn.Module):
    """
    Lightweight residual adapter applied to the frozen SAM2
    image-encoder features.

    SAM2 Hiera-Tiny produces three feature levels:

        Level 0: [B,  32, 256, 256]
        Level 1: [B,  64, 128, 128]
        Level 2: [B, 256,  64,  64]

    Each feature level has its own lightweight adapter.

    IMPORTANT:
        SAM2 encoder = FROZEN
        SAM2 mask decoder = FROZEN
        Only this adapter = TRAINABLE

    The adapter is residual:

        adapted = feature + learned_residual

    The final projection is initialized to zero so that the
    adapter initially behaves as an identity mapping.
    """

    def __init__(
        self,
        channels=(32, 64, 256),
        bottleneck=32
    ):
        super().__init__()

        self.channels = tuple(channels)
        self.bottleneck = bottleneck

        self.adapters = nn.ModuleList()

        for c in self.channels:

            block = nn.Sequential(

                # Down-project channels
                nn.Conv2d(
                    c,
                    bottleneck,
                    kernel_size=1,
                    bias=True
                ),

                nn.GELU(),

                # Project back to original channels
                nn.Conv2d(
                    bottleneck,
                    c,
                    kernel_size=1,
                    bias=True
                )
            )

            # Start as identity:
            #
            # adapted_feature = feature + 0
            #
            # Therefore the initial model behaves exactly
            # like the frozen SAM2 model.

            nn.init.zeros_(block[-1].weight)
            nn.init.zeros_(block[-1].bias)

            self.adapters.append(block)

    def forward(self, features):

        if not isinstance(features, (list, tuple)):
            raise TypeError(
                "Expected SAM2 encoder features as a list/tuple, "
                f"but received {type(features)}"
            )

        if len(features) != len(self.adapters):
            raise ValueError(
                f"Expected {len(self.adapters)} feature levels, "
                f"but received {len(features)}"
            )

        adapted_features = []

        for feature, adapter_block in zip(
            features,
            self.adapters
        ):

            adapted_feature = (
                feature + adapter_block(feature)
            )

            adapted_features.append(adapted_feature)

        return adapted_features


print("SAM2 multi-scale encoder adapter class ready.")

SAM2 multi-scale encoder adapter class ready.


In [37]:
# ============================================================
# SAM2 multi-scale feature configuration
# ============================================================

ADAPTER_BOTTLENECK = 32

# Your SAM2 Hiera-Tiny encoder returns 3 feature levels:
#
# Level 0: [B,  32, 256, 256]
# Level 1: [B,  64, 128, 128]
# Level 2: [B, 256,  64,  64]

FEATURE_CHANNELS = (32, 64, 256)

print("SAM2 feature levels:", len(FEATURE_CHANNELS))

for i, channels in enumerate(FEATURE_CHANNELS):
    print(f"Level {i}: {channels} channels")

print("Adapter bottleneck:", ADAPTER_BOTTLENECK)

SAM2 feature levels: 3
Level 0: 32 channels
Level 1: 64 channels
Level 2: 256 channels
Adapter bottleneck: 32


In [39]:
# ============================================================
# Parameter counts — Frozen SAM2 + Encoder Adapter
# ============================================================

_count_model, _ = create_sam2_from_decoder_checkpoint(
    VEGANN_SAM2_CHECKPOINT
)

# ------------------------------------------------------------
# SAM2 total parameters
# ------------------------------------------------------------

sam2_total_parameters = sum(
    p.numel()
    for p in _count_model.parameters()
)

# ------------------------------------------------------------
# Mask decoder parameters
# ------------------------------------------------------------

decoder_parameters = sum(
    p.numel()
    for p in _count_model.mask_decoder.parameters()
)

# ------------------------------------------------------------
# Encoder parameters
# ------------------------------------------------------------

encoder_parameters = (
    sam2_total_parameters - decoder_parameters
)

# ------------------------------------------------------------
# Multi-scale encoder adapter
# ------------------------------------------------------------

_probe_adapter = SAM2MultiScaleFeatureAdapter(
    channels=FEATURE_CHANNELS,
    bottleneck=ADAPTER_BOTTLENECK
).to(DEVICE)

adapter_parameters = sum(
    p.numel()
    for p in _probe_adapter.parameters()
)

# ------------------------------------------------------------
# Clean up
# ------------------------------------------------------------

del _count_model
del _probe_adapter

torch.cuda.empty_cache()

# ------------------------------------------------------------
# Print results
# ------------------------------------------------------------

print("=" * 70)
print("PARAMETER COUNT")
print("=" * 70)

print(
    f"SAM2 total parameters:        "
    f"{sam2_total_parameters:,}"
)

print(
    f"SAM2 encoder parameters:      "
    f"{encoder_parameters:,}"
)

print(
    f"Mask decoder parameters:      "
    f"{decoder_parameters:,}"
)

print(
    f"Encoder adapter parameters:   "
    f"{adapter_parameters:,}"
)

print("-" * 70)

print(
    f"Previous decoder-only setup:  "
    f"{decoder_parameters:,} trainable"
)

print(
    f"New adapter-only setup:       "
    f"{adapter_parameters:,} trainable"
)

print(
    f"Adapter overhead vs SAM2:     "
    f"{100 * adapter_parameters / sam2_total_parameters:.4f}%"
)

print("=" * 70)

[transformers] You are using a model of type `sam2_video` to instantiate a model of type `sam2`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Loading weights:   0%|          | 0/309 [00:00<?, ?it/s]

PARAMETER COUNT
SAM2 total parameters:        31,440,721
SAM2 encoder parameters:      27,225,612
Mask decoder parameters:      4,215,109
Encoder adapter parameters:   22,976
----------------------------------------------------------------------
Previous decoder-only setup:  4,215,109 trainable
New adapter-only setup:       22,976 trainable
Adapter overhead vs SAM2:     0.0731%


## 14. Encoder-adapter training / validation functions

In [53]:
# ============================================================
# TRAIN ONE EPOCH — ENCODER ADAPTER ONLY
# SAM2 + DECODER COMPLETELY FROZEN
# ============================================================

def train_one_epoch_encoder_adapter(
    model,
    adapter,
    loader,
    optimizer,
    scaler=None,
):

    # --------------------------------------------------------
    # Modes
    # --------------------------------------------------------

    model.eval()       # SAM2 completely frozen
    adapter.train()    # ONLY adapter trains

    # --------------------------------------------------------
    # HARD FREEZE SAM2
    # --------------------------------------------------------

    for p in model.parameters():
        p.requires_grad = False

    # Explicitly freeze decoder as well
    for p in model.mask_decoder.parameters():
        p.requires_grad = False

    # Adapter is the ONLY trainable component
    for p in adapter.parameters():
        p.requires_grad = True

    # --------------------------------------------------------
    # Safety checks
    # --------------------------------------------------------

    sam2_trainable = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    adapter_trainable = sum(
        p.numel()
        for p in adapter.parameters()
        if p.requires_grad
    )

    assert sam2_trainable == 0, (
        f"SAM2 is not completely frozen! "
        f"{sam2_trainable:,} parameters are trainable."
    )

    assert adapter_trainable > 0, (
        "Adapter has no trainable parameters."
    )

    # --------------------------------------------------------
    # Running statistics
    # --------------------------------------------------------

    running_loss = 0.0
    running_bce = 0.0
    running_dice = 0.0

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    for batch_index, batch in enumerate(loader):

        inputs = prepare_sam2_batch(batch)

        optimizer.zero_grad(set_to_none=True)

        # ====================================================
        # 1. FROZEN SAM2 IMAGE ENCODER
        # ====================================================

        with torch.no_grad():

            image_embeddings = model.get_image_embeddings(
                inputs["pixel_values"]
            )

        # ----------------------------------------------------
        # IMPORTANT:
        # image_embeddings is a LIST of feature tensors
        # ----------------------------------------------------

        if not isinstance(image_embeddings, (list, tuple)):
            raise TypeError(
                "Expected SAM2 image embeddings to be a "
                f"list/tuple, got {type(image_embeddings)}"
            )

        # ----------------------------------------------------
        # Check original encoder features
        # ----------------------------------------------------

        for level, feat in enumerate(image_embeddings):

            if not torch.isfinite(feat).all():

                raise RuntimeError(
                    f"Non-finite values detected in "
                    f"ORIGINAL image embedding at "
                    f"feature level {level}"
                )

        # ====================================================
        # 2. TRAINABLE ADAPTER
        # ====================================================

        adapted_embeddings = adapter(image_embeddings)

        # ----------------------------------------------------
        # Check adapted features
        # ----------------------------------------------------

        if not isinstance(adapted_embeddings, (list, tuple)):
            raise TypeError(
                "Expected adapter output to be a list/tuple, "
                f"got {type(adapted_embeddings)}"
            )

        for level, feat in enumerate(adapted_embeddings):

            if not torch.isfinite(feat).all():

                raise RuntimeError(
                    f"Non-finite values detected in "
                    f"ADAPTED embedding at feature level "
                    f"{level}, batch {batch_index + 1}"
                )

        # ====================================================
        # 3. FROZEN SAM2 DECODER
        # ====================================================

        outputs = model(
            image_embeddings=adapted_embeddings,
            input_boxes=inputs["input_boxes"],
            multimask_output=False,
        )

        # ====================================================
        # 4. MASK LOGITS
        # ====================================================

        logits = extract_single_mask_logits(
            outputs.pred_masks
        )

        if not torch.isfinite(logits).all():

            raise RuntimeError(
                f"Non-finite mask logits detected "
                f"at batch {batch_index + 1}"
            )

        # ====================================================
        # 5. TARGET MASK
        # ====================================================

        targets = inputs["labels"].float()

        while targets.ndim > 3:
            targets = targets[:, 0]

        targets = F.interpolate(
            targets.unsqueeze(1),
            size=logits.shape[-2:],
            mode="nearest",
        ).squeeze(1)

        targets = (targets > 0).float()

        # ====================================================
        # 6. LOSS
        # ====================================================

        loss, bce, dice = combined_segmentation_loss(
            logits,
            targets
        )

        if not torch.isfinite(loss):

            raise RuntimeError(
                f"Non-finite LOSS detected at "
                f"batch {batch_index + 1}\n"
                f"Loss: {loss}\n"
                f"BCE: {bce}\n"
                f"Dice: {dice}"
            )

        # ====================================================
        # 7. BACKWARD
        # ====================================================

        loss.backward()

        # ====================================================
        # 8. CHECK ADAPTER GRADIENTS
        # ====================================================

        total_grad_norm = 0.0

        for name, p in adapter.named_parameters():

            if p.grad is None:
                continue

            if not torch.isfinite(p.grad).all():

                raise RuntimeError(
                    f"Non-finite gradient detected in "
                    f"adapter parameter: {name} "
                    f"at batch {batch_index + 1}"
                )

            grad_norm = p.grad.detach().norm().item()

            total_grad_norm += grad_norm ** 2

        total_grad_norm = total_grad_norm ** 0.5

        # ----------------------------------------------------
        # Gradient clipping
        # ----------------------------------------------------

        torch.nn.utils.clip_grad_norm_(
            adapter.parameters(),
            max_norm=1.0
        )

        # ====================================================
        # 9. OPTIMIZER STEP
        # ====================================================

        optimizer.step()

        # ====================================================
        # 10. CHECK ADAPTER PARAMETERS AFTER UPDATE
        # ====================================================

        for name, p in adapter.named_parameters():

            if not torch.isfinite(p).all():

                raise RuntimeError(
                    f"Non-finite ADAPTER PARAMETER detected "
                    f"after optimizer step: {name} "
                    f"at batch {batch_index + 1}"
                )

        # ====================================================
        # 11. RUNNING STATISTICS
        # ====================================================

        running_loss += loss.item()
        running_bce += bce.item()
        running_dice += dice.item()

        # ====================================================
        # 12. PROGRESS
        # ====================================================

        if (batch_index + 1) % 100 == 0:

            avg_loss = (
                running_loss / (batch_index + 1)
            )

            avg_bce = (
                running_bce / (batch_index + 1)
            )

            avg_dice = (
                running_dice / (batch_index + 1)
            )

            print(
                f"Batch {batch_index + 1}/{len(loader)} | "
                f"Loss: {avg_loss:.4f} | "
                f"BCE: {avg_bce:.4f} | "
                f"Dice: {avg_dice:.4f} | "
                f"GradNorm: {total_grad_norm:.4f}"
            )

    # --------------------------------------------------------
    # Return epoch statistics
    # --------------------------------------------------------

    n = len(loader)

    return {
        "loss": running_loss / n,
        "bce": running_bce / n,
        "dice_loss": running_dice / n,
    }

## 15. Experiment config — all four datasets
**Check the ReLeaf/WeedsGalore checkpoint subfolder names below against what actually exists on disk before running the next cell:**
```python
print(sorted(p.name for p in DOMAIN_FT_ROOT.iterdir() if p.is_dir()))
```

In [41]:
print(sorted(p.name for p in DOMAIN_FT_ROOT.iterdir() if p.is_dir())
      if DOMAIN_FT_ROOT.exists() else "DOMAIN_FT_ROOT does not exist yet")

['All_Domains_EncoderAdapter', 'EWI1S_10_labels', 'EWI1S_25_labels', 'EWI1S_50_labels', 'EWI1S_5_labels', 'EWI1S_EncoderAdapter', 'EWI1S_Full', 'ReLeaf_100_labels', 'ReLeaf_10_labels', 'ReLeaf_25_labels', 'ReLeaf_50_labels', 'ReLeaf_5_labels', 'ReLeaf_EncoderAdapter', 'ReLeaf_Full', 'VegAnn_EncoderAdapter', 'VegAnn_ReLeaf_WeedsGalore_EWI1S', 'VegAnn_to_EWI1S', 'VegAnn_to_ReLeaf', 'VegAnn_to_WeedsGalore', 'WeedsGalore_100_labels', 'WeedsGalore_10_labels', 'WeedsGalore_25_labels', 'WeedsGalore_50_labels', 'WeedsGalore_5_labels', 'WeedsGalore_EncoderAdapter', 'WeedsGalore_Full', 'test_splits']


In [49]:
ADAPTER_DATASETS = ["VegAnn", "ReLeaf", "WeedsGalore", "EWI1S"]

ADAPTER_LEARNING_RATE = 1e-5
ADAPTER_WEIGHT_DECAY = 1e-4
ADAPTER_PATIENCE = 3

TOP_K = 32
BOX_MARGIN = 25

# VegAnn's train set is far larger than the other three (thousands vs.
# tens of images) — give it a smaller epoch budget instead of an
# accidental multi-hour run. Adjust to taste.
ADAPTER_NUM_EPOCHS_BY_DATASET = {
    "VegAnn": 3,
    "ReLeaf": 10,
    "WeedsGalore": 10,
    "EWI1S": 10,
}

# Existing decoder-only checkpoints from earlier domain fine-tuning.
# VegAnn's "decoder-only" checkpoint is the original VegAnn checkpoint
# itself (in-domain control, no fine-tuning needed).
# >>> EDIT these two paths if your folder names differ from the print()
#     above. <<<
DECODER_ONLY_CHECKPOINTS = {
    "VegAnn": VEGANN_SAM2_CHECKPOINT,
    "ReLeaf": DOMAIN_FT_ROOT / "VegAnn_to_ReLeaf" / "best_mask_decoder.pt",
    "WeedsGalore": DOMAIN_FT_ROOT / "VegAnn_to_WeedsGalore" / "best_mask_decoder.pt",
    "EWI1S": DOMAIN_FT_ROOT / "VegAnn_to_EWI1S" / "best_mask_decoder.pt",
}

ADAPTER_EXPERIMENT_DIRS = {
    name: DOMAIN_FT_ROOT / f"{name}_EncoderAdapter" for name in ADAPTER_DATASETS
}
for path in ADAPTER_EXPERIMENT_DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

ADAPTER_CHECKPOINTS = {
    name: ADAPTER_EXPERIMENT_DIRS[name] / "best_encoder_adapter.pt"
    for name in ADAPTER_DATASETS
}

for name in ADAPTER_DATASETS:
    print(f"\n{name}")
    print("  Train images:", len(datasets_train[name]))
    print("  Val images:  ", len(datasets_val[name]))
    print("  Decoder-only checkpoint:", DECODER_ONLY_CHECKPOINTS[name])
    print("  Adapter checkpoint:     ", ADAPTER_CHECKPOINTS[name])

_missing = [n for n, p in DECODER_ONLY_CHECKPOINTS.items() if not p.exists()]
if _missing:
    raise FileNotFoundError(
        f"Missing decoder-only checkpoint(s) for: {_missing}. "
        "Check the folder names above against DOMAIN_FT_ROOT's actual contents."
    )


VegAnn
  Train images: 2966
  Val images:   746
  Decoder-only checkpoint: /home/zahranadeem/thesis/sam2_vegann_checkpoints/best_sam2_vegann_mask_decoder.pt
  Adapter checkpoint:      /home/zahranadeem/thesis/sam2_domain_finetuning/VegAnn_EncoderAdapter/best_encoder_adapter.pt

ReLeaf
  Train images: 276
  Val images:   34
  Decoder-only checkpoint: /home/zahranadeem/thesis/sam2_domain_finetuning/VegAnn_to_ReLeaf/best_mask_decoder.pt
  Adapter checkpoint:      /home/zahranadeem/thesis/sam2_domain_finetuning/ReLeaf_EncoderAdapter/best_encoder_adapter.pt

WeedsGalore
  Train images: 124
  Val images:   15
  Decoder-only checkpoint: /home/zahranadeem/thesis/sam2_domain_finetuning/VegAnn_to_WeedsGalore/best_mask_decoder.pt
  Adapter checkpoint:      /home/zahranadeem/thesis/sam2_domain_finetuning/WeedsGalore_EncoderAdapter/best_encoder_adapter.pt

EWI1S
  Train images: 70
  Val images:   8
  Decoder-only checkpoint: /home/zahranadeem/thesis/sam2_domain_finetuning/VegAnn_to_EWI1S/best_mask

## 16. Train the encoder adapter on each dataset

In [47]:
# ============================================================
# DIAGNOSTIC — Check Adapter Numerical Stability
# ============================================================

adapter_model, _ = create_sam2_from_decoder_checkpoint(
    DECODER_ONLY_CHECKPOINTS["VegAnn"]
)

# Freeze EVERYTHING in SAM2
for p in adapter_model.parameters():
    p.requires_grad = False

adapter = SAM2MultiScaleFeatureAdapter(
    channels=FEATURE_CHANNELS,
    bottleneck=ADAPTER_BOTTLENECK
).to(DEVICE)

print("=" * 70)
print("INITIAL ADAPTER PARAMETER CHECK")
print("=" * 70)

for name, p in adapter.named_parameters():

    print(
        f"{name:50s} | "
        f"min={p.data.min().item():.6f} | "
        f"max={p.data.max().item():.6f} | "
        f"mean={p.data.mean().item():.6f} | "
        f"finite={torch.isfinite(p.data).all().item()}"
    )

print("\nAll adapter parameters finite:",
      all(
          torch.isfinite(p).all().item()
          for p in adapter.parameters()
      ))

[transformers] You are using a model of type `sam2_video` to instantiate a model of type `sam2`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Loading weights:   0%|          | 0/309 [00:00<?, ?it/s]

INITIAL ADAPTER PARAMETER CHECK
adapters.0.0.weight                                | min=-0.175653 | max=0.175729 | mean=-0.000891 | finite=True
adapters.0.0.bias                                  | min=-0.168779 | max=0.171479 | mean=0.005206 | finite=True
adapters.0.2.weight                                | min=0.000000 | max=0.000000 | mean=0.000000 | finite=True
adapters.0.2.bias                                  | min=0.000000 | max=0.000000 | mean=0.000000 | finite=True
adapters.1.0.weight                                | min=-0.124969 | max=0.124943 | mean=-0.001291 | finite=True
adapters.1.0.bias                                  | min=-0.118031 | max=0.114917 | mean=-0.014834 | finite=True
adapters.1.2.weight                                | min=0.000000 | max=0.000000 | mean=0.000000 | finite=True
adapters.1.2.bias                                  | min=0.000000 | max=0.000000 | mean=0.000000 | finite=True
adapters.2.0.weight                                | min=-0.062476 | max=

In [51]:
print("=" * 70)
print("ADAPTER NUMERICAL DIAGNOSTIC")
print("=" * 70)

adapter.eval()

with torch.no_grad():

    batch = next(iter(train_loader_adapter))
    inputs = prepare_sam2_batch(batch)

    image_embeddings = adapter_model.get_image_embeddings(
        inputs["pixel_values"]
    )

    print("\nImage embeddings:")
    print("  type:", type(image_embeddings))
    print("  number of feature levels:", len(image_embeddings))

    for i, feat in enumerate(image_embeddings):

        print(f"\n  Feature level {i}:")
        print("    type   :", type(feat))
        print("    shape  :", tuple(feat.shape))
        print("    min    :", feat.min().item())
        print("    max    :", feat.max().item())
        print("    mean   :", feat.mean().item())
        print("    std    :", feat.std().item())
        print("    finite :", torch.isfinite(feat).all().item())

    # ---------------------------------------------------------
    # Apply adapter
    # ---------------------------------------------------------

    adapted_embeddings = adapter(image_embeddings)

    print("\n" + "=" * 70)
    print("ADAPTED EMBEDDINGS")
    print("=" * 70)

    print("\nType:", type(adapted_embeddings))
    print("Number of feature levels:", len(adapted_embeddings))

    for i, feat in enumerate(adapted_embeddings):

        print(f"\n  Feature level {i}:")
        print("    shape  :", tuple(feat.shape))
        print("    min    :", feat.min().item())
        print("    max    :", feat.max().item())
        print("    mean   :", feat.mean().item())
        print("    std    :", feat.std().item())
        print("    finite :", torch.isfinite(feat).all().item())

    # ---------------------------------------------------------
    # Difference between original and adapted features
    # ---------------------------------------------------------

    print("\n" + "=" * 70)
    print("ADAPTER CHANGE")
    print("=" * 70)

    for i, (original, adapted) in enumerate(
        zip(image_embeddings, adapted_embeddings)
    ):

        difference = adapted - original

        print(f"\n  Feature level {i}:")
        print("    delta min  :", difference.min().item())
        print("    delta max  :", difference.max().item())
        print("    delta mean :", difference.mean().item())
        print("    delta std  :", difference.std().item())

ADAPTER NUMERICAL DIAGNOSTIC

Image embeddings:
  type: <class 'list'>
  number of feature levels: 3

  Feature level 0:
    type   : <class 'torch.Tensor'>
    shape  : (1, 32, 256, 256)
    min    : -0.9462965130805969
    max    : 0.6337892413139343
    mean   : -0.0025641086976975203
    std    : 0.10433022677898407
    finite : True

  Feature level 1:
    type   : <class 'torch.Tensor'>
    shape  : (1, 64, 128, 128)
    min    : -1.3333256244659424
    max    : 1.3149666786193848
    mean   : 0.028159355744719505
    std    : 0.19344446063041687
    finite : True

  Feature level 2:
    type   : <class 'torch.Tensor'>
    shape  : (1, 256, 64, 64)
    min    : -1.5782577991485596
    max    : 1.4990755319595337
    mean   : 0.008304236456751823
    std    : 0.26416319608688354
    finite : True

ADAPTED EMBEDDINGS

Type: <class 'list'>
Number of feature levels: 3

  Feature level 0:
    shape  : (1, 32, 256, 256)
    min    : -0.9462965130805969
    max    : 0.6337892413139343
 

In [54]:
# ============================================================
# TRAIN ENCODER ADAPTERS — DECODER FROZEN
# ============================================================
#
# For each dataset:
#
#   1. Load its EXISTING fine-tuned decoder checkpoint
#   2. Freeze the complete SAM2 model
#   3. Create a fresh multi-scale encoder adapter
#   4. Train ONLY the adapter
#   5. Keep the decoder checkpoint unchanged
#
# Datasets:
#   VegAnn
#   ReLeaf
#   WeedsGalore
#   EWI1S
#
# ============================================================

import math

# ------------------------------------------------------------
# IMPORTANT: conservative settings for numerical stability
# ------------------------------------------------------------

ADAPTER_LEARNING_RATE = 1e-5
ADAPTER_WEIGHT_DECAY = 1e-4

adapter_histories = {}
adapter_best_ious = {}


for dataset_name in ADAPTER_DATASETS:

    print("\n" + "#" * 70)
    print(f"# TRAINING ENCODER ADAPTER — {dataset_name}")
    print("#" * 70)

    # --------------------------------------------------------
    # Data loaders
    # --------------------------------------------------------

    train_loader_adapter = DataLoader(
        datasets_train[dataset_name],
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        collate_fn=sam2_collate_fn,
    )

    val_loader_adapter = DataLoader(
        datasets_val[dataset_name],
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        collate_fn=sam2_collate_fn,
    )

    print(
        f"Train samples: {len(datasets_train[dataset_name])}"
    )
    print(
        f"Val samples:   {len(datasets_val[dataset_name])}"
    )

    # --------------------------------------------------------
    # Load THIS DATASET'S existing decoder checkpoint
    # --------------------------------------------------------

    decoder_checkpoint = DECODER_ONLY_CHECKPOINTS[dataset_name]

    print(
        f"\nStarting decoder checkpoint:\n"
        f"  {decoder_checkpoint}"
    )

    adapter_model, _ = create_sam2_from_decoder_checkpoint(
        decoder_checkpoint
    )

    adapter_model = adapter_model.to(DEVICE)

    # --------------------------------------------------------
    # Freeze EVERYTHING in SAM2
    #
    # This freezes:
    #   - image encoder
    #   - prompt encoder
    #   - mask decoder
    #   - all other SAM2 parameters
    # --------------------------------------------------------

    adapter_model.eval()

    for p in adapter_model.parameters():
        p.requires_grad = False

    # Explicitly verify decoder is frozen
    for p in adapter_model.mask_decoder.parameters():
        p.requires_grad = False

    # --------------------------------------------------------
    # Create ONLY the trainable encoder adapter
    # --------------------------------------------------------

    adapter = SAM2MultiScaleFeatureAdapter(
        channels=FEATURE_CHANNELS,
        bottleneck=ADAPTER_BOTTLENECK
    ).to(DEVICE)

    adapter.train()

    for p in adapter.parameters():
        p.requires_grad = True

    # --------------------------------------------------------
    # Safety checks
    # --------------------------------------------------------

    decoder_trainable = sum(
        p.numel()
        for p in adapter_model.mask_decoder.parameters()
        if p.requires_grad
    )

    sam2_trainable = sum(
        p.numel()
        for p in adapter_model.parameters()
        if p.requires_grad
    )

    adapter_trainable = sum(
        p.numel()
        for p in adapter.parameters()
        if p.requires_grad
    )

    print("\nTrainable parameter check:")
    print(
        f"  SAM2 trainable parameters:     "
        f"{sam2_trainable:,}"
    )
    print(
        f"  Decoder trainable parameters:  "
        f"{decoder_trainable:,}"
    )
    print(
        f"  Adapter trainable parameters:  "
        f"{adapter_trainable:,}"
    )

    # These MUST be zero
    assert sam2_trainable == 0, (
        "ERROR: Some SAM2 parameters are still trainable!"
    )

    assert decoder_trainable == 0, (
        "ERROR: Mask decoder is still trainable!"
    )

    assert adapter_trainable > 0, (
        "ERROR: Adapter has no trainable parameters!"
    )

    # --------------------------------------------------------
    # Initial adapter numerical check
    # --------------------------------------------------------

    print("\nChecking initial adapter parameters...")

    all_adapter_finite = True

    for name, p in adapter.named_parameters():

        finite = torch.isfinite(p).all().item()

        print(
            f"{name:40s} | "
            f"min={p.min().item():.6f} | "
            f"max={p.max().item():.6f} | "
            f"mean={p.mean().item():.6f} | "
            f"finite={finite}"
        )

        if not finite:
            all_adapter_finite = False

    print(
        f"\nAll adapter parameters finite: "
        f"{all_adapter_finite}"
    )

    assert all_adapter_finite, (
        "ERROR: Adapter contains NaN/Inf before training!"
    )

    # --------------------------------------------------------
    # Optimizer
    #
    # ONLY adapter parameters
    # --------------------------------------------------------

    optimizer = torch.optim.AdamW(
        adapter.parameters(),
        lr=ADAPTER_LEARNING_RATE,
        weight_decay=ADAPTER_WEIGHT_DECAY,
    )

    # --------------------------------------------------------
    # AMP DISABLED FOR DEBUGGING
    #
    # Once training is stable, we can turn AMP back on.
    # --------------------------------------------------------

    scaler = None

    print("\nOptimizer:")
    print(f"  Learning rate: {ADAPTER_LEARNING_RATE}")
    print(f"  Weight decay:  {ADAPTER_WEIGHT_DECAY}")
    print("  AMP:            DISABLED")

    # --------------------------------------------------------
    # History
    # --------------------------------------------------------

    history = {
        "epoch": [],
        "train_loss": [],
        "train_bce": [],
        "train_dice_loss": [],
        "val_loss": [],
        "mean_iou": [],
        "mean_dice": [],
        "global_iou": [],
        "global_dice": [],
    }

    best_iou = -1.0
    epochs_without_improvement = 0

    num_epochs = ADAPTER_NUM_EPOCHS_BY_DATASET.get(
        dataset_name,
        10
    )

    # ========================================================
    # Training loop
    # ========================================================

    for epoch in range(num_epochs):

        print("\n" + "=" * 70)
        print(
            f"{dataset_name} ENCODER ADAPTER — "
            f"EPOCH {epoch + 1}/{num_epochs}"
        )
        print("=" * 70)

        # ----------------------------------------------------
        # Make absolutely sure SAM2 remains frozen
        # ----------------------------------------------------

        adapter_model.eval()
        adapter.train()

        for p in adapter_model.parameters():
            assert not p.requires_grad, (
                "ERROR: SAM2 parameter became trainable!"
            )

        for p in adapter.parameters():
            p.requires_grad = True

        # ----------------------------------------------------
        # Train ONLY adapter
        # ----------------------------------------------------

        train_results = train_one_epoch_encoder_adapter(
            adapter_model,
            adapter,
            train_loader_adapter,
            optimizer,
            scaler
        )

        # ----------------------------------------------------
        # Validation
        # ----------------------------------------------------

        val_results = validate_encoder_adapter(
            adapter_model,
            adapter,
            val_loader_adapter,
            MASK_THRESHOLD
        )

        # ----------------------------------------------------
        # Check validation results are finite
        # ----------------------------------------------------

        for key, value in val_results.items():

            if isinstance(value, (float, int)):

                if not math.isfinite(float(value)):

                    raise RuntimeError(
                        f"Non-finite validation result: "
                        f"{key} = {value}"
                    )

        # ----------------------------------------------------
        # Save history
        # ----------------------------------------------------

        history["epoch"].append(epoch + 1)

        history["train_loss"].append(
            train_results["loss"]
        )

        history["train_bce"].append(
            train_results["bce"]
        )

        history["train_dice_loss"].append(
            train_results["dice_loss"]
        )

        history["val_loss"].append(
            val_results["loss"]
        )

        history["mean_iou"].append(
            val_results["mean_iou"]
        )

        history["mean_dice"].append(
            val_results["mean_dice"]
        )

        history["global_iou"].append(
            val_results["global_iou"]
        )

        history["global_dice"].append(
            val_results["global_dice"]
        )

        # ----------------------------------------------------
        # Print results
        # ----------------------------------------------------

        print(
            f"\nTrain Loss: "
            f"{train_results['loss']:.4f}"
        )

        print(
            f"Val Loss:   "
            f"{val_results['loss']:.4f}"
        )

        print(
            f"Mean IoU:   "
            f"{100 * val_results['mean_iou']:.2f}%"
        )

        print(
            f"Mean Dice:  "
            f"{100 * val_results['mean_dice']:.2f}%"
        )

        print(
            f"Global IoU: "
            f"{100 * val_results['global_iou']:.2f}%"
        )

        print(
            f"Global Dice:"
            f" {100 * val_results['global_dice']:.2f}%"
        )

        # ----------------------------------------------------
        # Best checkpoint
        # ----------------------------------------------------

        current_iou = val_results["mean_iou"]

        if not math.isfinite(current_iou):

            print(
                "WARNING: Validation IoU is non-finite. "
                "Checkpoint NOT saved."
            )

        elif current_iou > best_iou:

            best_iou = current_iou
            epochs_without_improvement = 0

            torch.save(
                {
                    "experiment_name":
                        f"{dataset_name}_EncoderAdapter",

                    "epoch":
                        epoch + 1,

                    "model_name":
                        MODEL_NAME,

                    # Actual decoder used
                    "starting_decoder_checkpoint":
                        str(decoder_checkpoint),

                    "train_dataset":
                        dataset_name,

                    "train_samples":
                        len(datasets_train[dataset_name]),

                    "val_samples":
                        len(datasets_val[dataset_name]),

                    "learning_rate":
                        ADAPTER_LEARNING_RATE,

                    "weight_decay":
                        ADAPTER_WEIGHT_DECAY,

                    "adapter_channels":
                        FEATURE_CHANNELS,

                    "adapter_bottleneck":
                        ADAPTER_BOTTLENECK,

                    # Frozen decoder state
                    "mask_decoder_state_dict":
                        adapter_model.mask_decoder.state_dict(),

                    # Trainable adapter
                    "encoder_adapter_state_dict":
                        adapter.state_dict(),

                    "best_mean_iou":
                        best_iou,

                    "history":
                        history,
                },
                ADAPTER_CHECKPOINTS[dataset_name]
            )

            print(
                "\n*** BEST "
                f"{dataset_name} ENCODER-ADAPTER "
                "CHECKPOINT SAVED ***"
            )

        else:

            epochs_without_improvement += 1

            print(
                f"No improvement for "
                f"{epochs_without_improvement} epoch(s)."
            )

        # ----------------------------------------------------
        # Save training history
        # ----------------------------------------------------

        pd.DataFrame(history).to_csv(
            ADAPTER_EXPERIMENT_DIRS[dataset_name]
            / "training_history.csv",
            index=False
        )

        # ----------------------------------------------------
        # Early stopping
        # ----------------------------------------------------

        if (
            epochs_without_improvement
            >= ADAPTER_PATIENCE
        ):

            print(
                f"\n{dataset_name} encoder-adapter "
                "early stopping triggered."
            )

            break

    # --------------------------------------------------------
    # Store results
    # --------------------------------------------------------

    adapter_histories[dataset_name] = pd.DataFrame(
        history
    )

    adapter_best_ious[dataset_name] = best_iou

    # --------------------------------------------------------
    # Clean up
    # --------------------------------------------------------

    del adapter_model
    del adapter
    del optimizer

    torch.cuda.empty_cache()

    print(
        f"\n{dataset_name} best validation IoU: "
        f"{100 * best_iou:.2f}%"
    )


# ============================================================
# Final summary
# ============================================================

print("\n\n" + "=" * 70)
print("ALL ENCODER-ADAPTER TRAINING COMPLETE")
print("=" * 70)

for name, iou in adapter_best_ious.items():

    print(
        f"  {name:15s}: "
        f"best val IoU = {100 * iou:.2f}%"
    )

[transformers] You are using a model of type `sam2_video` to instantiate a model of type `sam2`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.



######################################################################
# TRAINING ENCODER ADAPTER — VegAnn
######################################################################
Train samples: 2966
Val samples:   746

Starting decoder checkpoint:
  /home/zahranadeem/thesis/sam2_vegann_checkpoints/best_sam2_vegann_mask_decoder.pt


Loading weights:   0%|          | 0/309 [00:00<?, ?it/s]


Trainable parameter check:
  SAM2 trainable parameters:     0
  Decoder trainable parameters:  0
  Adapter trainable parameters:  22,976

Checking initial adapter parameters...
adapters.0.0.weight                      | min=-0.176750 | max=0.176765 | mean=-0.006589 | finite=True
adapters.0.0.bias                        | min=-0.162330 | max=0.172645 | mean=-0.020827 | finite=True
adapters.0.2.weight                      | min=0.000000 | max=0.000000 | mean=0.000000 | finite=True
adapters.0.2.bias                        | min=0.000000 | max=0.000000 | mean=0.000000 | finite=True
adapters.1.0.weight                      | min=-0.124988 | max=0.124988 | mean=-0.002721 | finite=True
adapters.1.0.bias                        | min=-0.119327 | max=0.119643 | mean=-0.000184 | finite=True
adapters.1.2.weight                      | min=0.000000 | max=0.000000 | mean=0.000000 | finite=True
adapters.1.2.bias                        | min=0.000000 | max=0.000000 | mean=0.000000 | finite=True
adapte

[transformers] You are using a model of type `sam2_video` to instantiate a model of type `sam2`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.



Train Loss: 0.1062
Val Loss:   0.1183
Mean IoU:   87.38%
Mean Dice:  92.83%
Global IoU: 90.71%
Global Dice: 95.13%

*** BEST VegAnn ENCODER-ADAPTER CHECKPOINT SAVED ***

VegAnn best validation IoU: 87.38%

######################################################################
# TRAINING ENCODER ADAPTER — ReLeaf
######################################################################
Train samples: 276
Val samples:   34

Starting decoder checkpoint:
  /home/zahranadeem/thesis/sam2_domain_finetuning/VegAnn_to_ReLeaf/best_mask_decoder.pt


... (6 outputs left out)

[transformers] You are using a model of type `sam2_video` to instantiate a model of type `sam2`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.



Train Loss: 0.1387
Val Loss:   0.1474
Mean IoU:   65.78%
Mean Dice:  79.08%
Global IoU: 70.83%
Global Dice: 82.93%

*** BEST WeedsGalore ENCODER-ADAPTER CHECKPOINT SAVED ***

WeedsGalore best validation IoU: 65.78%

######################################################################
# TRAINING ENCODER ADAPTER — EWI1S
######################################################################
Train samples: 70
Val samples:   8

Starting decoder checkpoint:
  /home/zahranadeem/thesis/sam2_domain_finetuning/VegAnn_to_EWI1S/best_mask_decoder.pt


Loading weights:   0%|          | 0/309 [00:00<?, ?it/s]


Trainable parameter check:
  SAM2 trainable parameters:     0
  Decoder trainable parameters:  0
  Adapter trainable parameters:  22,976

Checking initial adapter parameters...
adapters.0.0.weight                      | min=-0.176758 | max=0.176345 | mean=-0.000442 | finite=True
adapters.0.0.bias                        | min=-0.159592 | max=0.166356 | mean=-0.004197 | finite=True
adapters.0.2.weight                      | min=0.000000 | max=0.000000 | mean=0.000000 | finite=True
adapters.0.2.bias                        | min=0.000000 | max=0.000000 | mean=0.000000 | finite=True
adapters.1.0.weight                      | min=-0.125000 | max=0.124815 | mean=-0.001427 | finite=True
adapters.1.0.bias                        | min=-0.119513 | max=0.120951 | mean=-0.007281 | finite=True
adapters.1.2.weight                      | min=0.000000 | max=0.000000 | mean=0.000000 | finite=True
adapters.1.2.bias                        | min=0.000000 | max=0.000000 | mean=0.000000 | finite=True
adapte

## 17. GT-box comparison: decoder-only vs. decoder + adapter, all datasets

In [56]:
# ============================================================
# COMPARE:
#   1. Decoder-only
#   2. Decoder + Encoder Adapter
#
# IMPORTANT:
# The adapter experiment uses:
#   Frozen Encoder
#   Trainable Adapter
#   Frozen Decoder during training
#
# Here we simply LOAD the trained adapter and evaluate it.
# ============================================================

gt_box_comparison_rows = []


for dataset_name in ADAPTER_DATASETS:

    print("\n" + "=" * 70)
    print(f"EVALUATING {dataset_name}")
    print("=" * 70)

    # --------------------------------------------------------
    # Validation loader
    # --------------------------------------------------------

    val_loader = DataLoader(
        datasets_val[dataset_name],
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        collate_fn=sam2_collate_fn,
    )

    # ========================================================
    # 1. DECODER-ONLY BASELINE
    # ========================================================

    print("\n[1/2] Evaluating decoder-only baseline...")

    baseline_model, _ = create_sam2_from_decoder_checkpoint(
        DECODER_ONLY_CHECKPOINTS[dataset_name]
    )

    baseline_model = baseline_model.to(DEVICE)

    baseline_val = validate(
        baseline_model,
        val_loader,
        MASK_THRESHOLD
    )

    decoder_params = sum(
        p.numel()
        for p in baseline_model.mask_decoder.parameters()
    )

    gt_box_comparison_rows.append({
        "dataset": dataset_name,
        "method": "Decoder-only",
        "trainable_parameters": decoder_params,
        "IoU_%": 100 * baseline_val["mean_iou"],
        "Dice_%": 100 * baseline_val["mean_dice"],
    })

    print(
        f"Decoder-only: "
        f"IoU={100 * baseline_val['mean_iou']:.2f}% | "
        f"Dice={100 * baseline_val['mean_dice']:.2f}%"
    )

    del baseline_model
    torch.cuda.empty_cache()

    # ========================================================
    # 2. DECODER + ENCODER ADAPTER
    # ========================================================

    print("\n[2/2] Evaluating encoder-adapter model...")

    # --------------------------------------------------------
    # Load base SAM2 architecture
    # --------------------------------------------------------

    adapter_eval_model, _ = create_sam2_from_decoder_checkpoint(
        DECODER_ONLY_CHECKPOINTS[dataset_name]
    )

    adapter_eval_model = adapter_eval_model.to(DEVICE)

    # --------------------------------------------------------
    # Load trained adapter checkpoint
    # --------------------------------------------------------

    ckpt = torch.load(
        ADAPTER_CHECKPOINTS[dataset_name],
        map_location="cpu",
        weights_only=False
    )

    print("\nCheckpoint information:")
    print(
        "  Adapter channels:",
        ckpt["adapter_channels"]
    )
    print(
        "  Adapter bottleneck:",
        ckpt["adapter_bottleneck"]
    )

    # --------------------------------------------------------
    # Load the SAME decoder state saved with adapter
    # --------------------------------------------------------

    adapter_eval_model.mask_decoder.load_state_dict(
        ckpt["mask_decoder_state_dict"],
        strict=True
    )

    # --------------------------------------------------------
    # Create the CORRECT adapter class
    # --------------------------------------------------------

    adapter_eval = SAM2MultiScaleFeatureAdapter(
        channels=ckpt["adapter_channels"],
        bottleneck=ckpt["adapter_bottleneck"]
    ).to(DEVICE)

    # --------------------------------------------------------
    # Load adapter weights
    # --------------------------------------------------------

    adapter_eval.load_state_dict(
        ckpt["encoder_adapter_state_dict"],
        strict=True
    )

    # --------------------------------------------------------
    # Freeze everything for evaluation
    # --------------------------------------------------------

    adapter_eval_model.eval()
    adapter_eval.eval()

    for p in adapter_eval_model.parameters():
        p.requires_grad = False

    for p in adapter_eval.parameters():
        p.requires_grad = False

    # --------------------------------------------------------
    # Validate
    # --------------------------------------------------------

    adapter_val = validate_encoder_adapter(
        adapter_eval_model,
        adapter_eval,
        val_loader,
        MASK_THRESHOLD
    )

    adapter_params = sum(
        p.numel()
        for p in adapter_eval.parameters()
    )

    gt_box_comparison_rows.append({
        "dataset": dataset_name,
        "method": "Decoder + Encoder Adapter",
        "trainable_parameters": adapter_params,
        "IoU_%": 100 * adapter_val["mean_iou"],
        "Dice_%": 100 * adapter_val["mean_dice"],
    })

    print(
        f"Decoder + Adapter: "
        f"IoU={100 * adapter_val['mean_iou']:.2f}% | "
        f"Dice={100 * adapter_val['mean_dice']:.2f}%"
    )

    # --------------------------------------------------------
    # Cleanup
    # --------------------------------------------------------

    del adapter_eval_model
    del adapter_eval
    del ckpt

    torch.cuda.empty_cache()


# ============================================================
# FINAL COMPARISON TABLE
# ============================================================

gt_box_comparison_df = pd.DataFrame(
    gt_box_comparison_rows
)

print("\n\n" + "=" * 70)
print("DECODER-ONLY vs ENCODER-ADAPTER")
print("=" * 70)

display(
    gt_box_comparison_df.round(3)
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

comparison_path = (
    DOMAIN_FT_ROOT
    / "GT_BOX_decoder_vs_adapter_ALL_DATASETS.csv"
)

gt_box_comparison_df.to_csv(
    comparison_path,
    index=False
)

print(
    f"\nSaved comparison to:\n"
    f"{comparison_path}"
)

[transformers] You are using a model of type `sam2_video` to instantiate a model of type `sam2`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.



EVALUATING VegAnn

[1/2] Evaluating decoder-only baseline...


Loading weights:   0%|          | 0/309 [00:00<?, ?it/s]

[transformers] You are using a model of type `sam2_video` to instantiate a model of type `sam2`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Decoder-only: IoU=87.25% | Dice=92.75%

[2/2] Evaluating encoder-adapter model...


Loading weights:   0%|          | 0/309 [00:00<?, ?it/s]

... (20 outputs left out)

Loading weights:   0%|          | 0/309 [00:00<?, ?it/s]


Checkpoint information:
  Adapter channels: (32, 64, 256)
  Adapter bottleneck: 32
Decoder + Adapter: IoU=33.36% | Dice=46.05%


DECODER-ONLY vs ENCODER-ADAPTER


,dataset,method,trainable_parameters,IoU_%,Dice_%
0,VegAnn,Decoder-only,4215109,87.248,92.748
1,VegAnn,Decoder + Encoder Adapter,22976,87.378,92.826
2,ReLeaf,Decoder-only,4215109,86.987,92.825
3,ReLeaf,Decoder + Encoder Adapter,22976,87.689,93.261
4,WeedsGalore,Decoder-only,4215109,65.546,78.919
5,WeedsGalore,Decoder + Encoder Adapter,22976,65.775,79.081
6,EWI1S,Decoder-only,4215109,29.225,40.815
7,EWI1S,Decoder + Encoder Adapter,22976,33.358,46.051



Saved comparison to:
/home/zahranadeem/thesis/sam2_domain_finetuning/GT_BOX_decoder_vs_adapter_ALL_DATASETS.csv


In [61]:
# ============================================================
# CALCULATE DECODER-ONLY vs ENCODER-ADAPTER IMPROVEMENTS
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# Separate the two methods
# ------------------------------------------------------------

decoder_df = (
    gt_box_comparison_df[
        gt_box_comparison_df["method"] == "Decoder-only"
    ]
    [["dataset", "IoU_%", "Dice_%"]]
    .rename(columns={
        "IoU_%": "Decoder_IoU_%",
        "Dice_%": "Decoder_Dice_%"
    })
)

adapter_df = (
    gt_box_comparison_df[
        gt_box_comparison_df["method"] == "Decoder + Encoder Adapter"
    ]
    [["dataset", "IoU_%", "Dice_%"]]
    .rename(columns={
        "IoU_%": "Adapter_IoU_%",
        "Dice_%": "Adapter_Dice_%"
    })
)

# ------------------------------------------------------------
# Merge
# ------------------------------------------------------------

comparison = decoder_df.merge(
    adapter_df,
    on="dataset",
    how="inner"
)

# ------------------------------------------------------------
# Absolute improvement
# ------------------------------------------------------------

comparison["IoU_Improvement_pp"] = (
    comparison["Adapter_IoU_%"]
    - comparison["Decoder_IoU_%"]
)

comparison["Dice_Improvement_pp"] = (
    comparison["Adapter_Dice_%"]
    - comparison["Decoder_Dice_%"]
)

# ------------------------------------------------------------
# Relative improvement (%)
#
# Formula:
# (new - old) / old * 100
# ------------------------------------------------------------

comparison["IoU_Relative_Improvement_%"] = (
    (
        comparison["Adapter_IoU_%"]
        - comparison["Decoder_IoU_%"]
    )
    / comparison["Decoder_IoU_%"]
) * 100

comparison["Dice_Relative_Improvement_%"] = (
    (
        comparison["Adapter_Dice_%"]
        - comparison["Decoder_Dice_%"]
    )
    / comparison["Decoder_Dice_%"]
) * 100

# ------------------------------------------------------------
# Round for display
# ------------------------------------------------------------

comparison_display = comparison.round({
    "Decoder_IoU_%": 3,
    "Adapter_IoU_%": 3,
    "IoU_Improvement_pp": 3,
    "IoU_Relative_Improvement_%": 3,
    "Decoder_Dice_%": 3,
    "Adapter_Dice_%": 3,
    "Dice_Improvement_pp": 3,
    "Dice_Relative_Improvement_%": 3,
})

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("=" * 100)
print("DECODER-ONLY vs ENCODER-ADAPTER IMPROVEMENT")
print("=" * 100)

display(comparison_display)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

improvement_path = (
    DOMAIN_FT_ROOT
    / "Decoder_vs_EncoderAdapter_Improvement_ALL_DATASETS.csv"
)

comparison_display.to_csv(
    improvement_path,
    index=False
)

print(f"\nSaved to:\n{improvement_path}")

DECODER-ONLY vs ENCODER-ADAPTER IMPROVEMENT


,dataset,Decoder_IoU_%,Decoder_Dice_%,Adapter_IoU_%,Adapter_Dice_%,IoU_Improvement_pp,Dice_Improvement_pp,IoU_Relative_Improvement_%,Dice_Relative_Improvement_%
0,VegAnn,87.248,92.748,87.378,92.826,0.130,0.078,0.148,0.084
1,ReLeaf,86.987,92.825,87.689,93.261,0.702,0.436,0.807,0.470
2,WeedsGalore,65.546,78.919,65.775,79.081,0.230,0.162,0.351,0.206
3,EWI1S,29.225,40.815,33.358,46.051,4.133,5.235,14.143,12.827



Saved to:
/home/zahranadeem/thesis/sam2_domain_finetuning/Decoder_vs_EncoderAdapter_Improvement_ALL_DATASETS.csv


## 18. AgriCLIP + DINO setup
Loads the AgriCLIP-aligned DINO model used to generate boxes for the final pipeline evaluation (as opposed to GT boxes used for training/validation above). Requires internet access for `torch.hub.load` the first time it runs, and the AgriCLIP repo present at `AGRICLIP_PATH`.

In [62]:
AGRICLIP_PATH = (
    "/home/zahranadeem/thesis/agriclip_sam2/AgriCLIP/AgriCLIP_alignment"
)
if AGRICLIP_PATH not in sys.path:
    sys.path.insert(0, AGRICLIP_PATH)

if not Path(AGRICLIP_PATH).exists():
    raise FileNotFoundError(
        f"AgriCLIP repo not found at: {AGRICLIP_PATH}\n"
        "Fix AGRICLIP_PATH before continuing — everything below this "
        "cell depends on importing from that repo."
    )

from TextToConcept import TextToConcept
from LinearAligner import LinearAligner
from torchvision import transforms

ALIGNER_PATH = Path(
    "/home/zahranadeem/thesis/agriclip_sam2/AgriCLIP/"
    "Weights/Aligned_Models/Agri_Dino_aligner_DDEF_CDEF.pth"
)
if not ALIGNER_PATH.exists():
    raise FileNotFoundError(f"AgriCLIP aligner weights not found:\n{ALIGNER_PATH}")

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

dino_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

dino_model = torch.hub.load("facebookresearch/dino:main", "dino_resnet50", pretrained=True)
dino_model.eval().to(device)
for p in dino_model.parameters():
    p.requires_grad = False

text_to_concept = TextToConcept(dino_model, "dino_resnet50")
text_to_concept.load_linear_aligner(str(ALIGNER_PATH))

aligner = text_to_concept.linear_aligner

print("AgriCLIP aligner loaded.")
print("Aligner type:", type(aligner))
print("W shape:", tuple(aligner.W.shape))
print("b shape:", tuple(aligner.b.shape))

Using cache found in /home/zahranadeem/.cache/torch/hub/facebookresearch_dino_main


AgriCLIP aligner loaded.
Aligner type: <class 'LinearAligner.LinearAligner'>
W shape: (512, 2048)
b shape: (512,)


In [86]:
captured_feature_map = None


def capture_layer4_output(module, inputs, output):
    global captured_feature_map
    captured_feature_map = output.detach()


hook_handle = dino_model.layer4.register_forward_hook(capture_layer4_output)


@torch.no_grad()
# ============================================================
# FIXED DINO PATCH FEATURE EXTRACTION
# ============================================================

def extract_dino_patch_features(image):
    """
    Extract spatial DINO ResNet50 layer4 features.

    DINO layer4 output:
        [1, 2048, 7, 7]

    We convert this to:
        [49, 2048]

    where each row corresponds to one spatial patch.

    Returns:
        patch_features : Tensor [49, 2048]
        grid_h         : 7
        grid_w         : 7
    """

    # --------------------------------------------------------
    # Ensure PIL image
    # --------------------------------------------------------
    if not isinstance(image, Image.Image):
        image = Image.fromarray(np.array(image))

    # --------------------------------------------------------
    # Apply the same transform used by the working DINO test
    # --------------------------------------------------------
    x = dino_transform(image).unsqueeze(0).to(device)

    captured = {}

    # --------------------------------------------------------
    # Hook the ACTUAL dino_model.layer4
    # --------------------------------------------------------
    def hook_fn(module, inputs, output):
        captured["layer4"] = output

    handle = dino_model.layer4.register_forward_hook(hook_fn)

    try:

        with torch.no_grad():
            _ = dino_model(x)

    finally:
        handle.remove()

    # --------------------------------------------------------
    # Verify feature was captured
    # --------------------------------------------------------
    if "layer4" not in captured:
        raise RuntimeError(
            "The DINO layer4 hook did not capture a feature map."
        )

    features = captured["layer4"]

    # --------------------------------------------------------
    # Check numerical validity
    # --------------------------------------------------------
    if not torch.isfinite(features).all():
        raise RuntimeError(
            "DINO layer4 feature map contains NaN or Inf values."
        )

    # --------------------------------------------------------
    # Expected shape:
    # [1, 2048, 7, 7]
    # --------------------------------------------------------
    if features.ndim != 4:
        raise RuntimeError(
            f"Unexpected DINO feature shape: {tuple(features.shape)}"
        )

    batch_size, channels, grid_h, grid_w = features.shape

    if batch_size != 1:
        raise RuntimeError(
            f"Expected batch size 1, got {batch_size}"
        )

    # --------------------------------------------------------
    # Convert:
    #
    # [1, 2048, 7, 7]
    #
    # -> [2048, 49]
    #
    # -> [49, 2048]
    # --------------------------------------------------------
    patch_features = features.squeeze(0)
    patch_features = patch_features.flatten(1).transpose(0, 1)

    # --------------------------------------------------------
    # Final sanity check
    # --------------------------------------------------------
    if not torch.isfinite(patch_features).all():
        raise RuntimeError(
            "Flattened DINO patch features contain NaN/Inf."
        )

    return patch_features, grid_h, grid_w


print("✓ Replaced extract_dino_patch_features()")


@torch.no_grad()
def align_patch_features(patch_features):
    aligned_features = aligner.get_aligned_representation(patch_features.float().to(DEVICE))
    return F.normalize(aligned_features, dim=1)

print("DINO feature extraction ready.")

✓ Replaced extract_dino_patch_features()
DINO feature extraction ready.


In [59]:
POSITIVE_PROMPTS = [
    "vegetation", "green vegetation", "crop plants",
    "crop vegetation", "agricultural plants", "green plants",
]
NEGATIVE_PROMPTS = [
    "soil", "bare soil", "ground", "bare ground",
    "non-vegetated ground", "agricultural soil",
]


@torch.no_grad()
def encode_prompt_set(prompts):
    embeddings = []
    for prompt in prompts:
        embedding = text_to_concept.encode_text([[prompt]])
        if isinstance(embedding, np.ndarray):
            embedding = torch.from_numpy(embedding)
        embedding = embedding.detach().float().to(device).reshape(-1)
        embeddings.append(embedding)
    embeddings = F.normalize(torch.stack(embeddings), dim=1)
    mean_embedding = F.normalize(embeddings.mean(dim=0), dim=0)
    return embeddings, mean_embedding


positive_text_embeddings, positive_embedding = encode_prompt_set(POSITIVE_PROMPTS)
negative_text_embeddings, negative_embedding = encode_prompt_set(NEGATIVE_PROMPTS)
vegetation_embedding = F.normalize(positive_embedding - negative_embedding, dim=0)

print("Positive embeddings:", positive_text_embeddings.shape)
print("Negative embeddings:", negative_text_embeddings.shape)
print("Final vegetation embedding:", vegetation_embedding.shape)

Positive embeddings: torch.Size([6, 512])
Negative embeddings: torch.Size([6, 512])
Final vegetation embedding: torch.Size([512])


In [60]:
@torch.no_grad()
def generate_agriclip_box(image, top_k=32, margin=25):
    patch_features, grid_h, grid_w = extract_dino_patch_features(image)
    aligned_features = align_patch_features(patch_features)
    patch_scores = aligned_features @ vegetation_embedding
    scores = patch_scores.cpu().numpy()
    topk_indices = np.argsort(scores)[::-1][:top_k]

    W, H = image.size
    patch_h = H / grid_h
    patch_w = W / grid_w

    boxes = []
    for idx in topk_indices:
        r = idx // grid_w
        c = idx % grid_w
        boxes.append([
            int(c * patch_w), int(r * patch_h),
            int((c + 1) * patch_w), int((r + 1) * patch_h),
        ])
    boxes = np.asarray(boxes)

    xmin = max(0, boxes[:, 0].min() - margin)
    ymin = max(0, boxes[:, 1].min() - margin)
    xmax = min(W - 1, boxes[:, 2].max() + margin)
    ymax = min(H - 1, boxes[:, 3].max() + margin)

    return np.array([xmin, ymin, xmax, ymax], dtype=np.int32), boxes, scores, topk_indices


sam2_processor = Sam2Processor.from_pretrained(MODEL_NAME)
print("AgriCLIP + DINO pipeline ready.")

AgriCLIP + DINO pipeline ready.


## 19. SAM2 multi-mask predictors (decoder-only and adapter)

In [63]:
@torch.no_grad()
def sam2_predict_box_multimask(model, image, box, threshold=0.5):
    inputs = sam2_processor(images=image, input_boxes=[[box.tolist()]], return_tensors="pt")
    inputs = {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in inputs.items()}

    outputs = model(**inputs, multimask_output=True)
    best_idx = outputs.iou_scores[0, 0].argmax().item()
    logits = outputs.pred_masks[0, 0, best_idx]
    probability = torch.sigmoid(logits)

    height = inputs["original_sizes"][0, 0].item()
    width = inputs["original_sizes"][0, 1].item()
    probability = F.interpolate(
        probability[None, None], size=(height, width), mode="bilinear", align_corners=False
    )[0, 0]

    prediction = (probability > threshold).cpu().numpy().astype(np.uint8)
    confidence = outputs.iou_scores[0, 0, best_idx].float().item()
    return prediction, confidence, best_idx


@torch.no_grad()
def adapter_sam2_predict_box_multimask(model, adapter, image, box, threshold=0.5):
    inputs = sam2_processor(images=image, input_boxes=[[box.tolist()]], return_tensors="pt")
    inputs = {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in inputs.items()}

    image_embeddings = model.get_image_embeddings(inputs["pixel_values"])
    adapted_embeddings = adapter(image_embeddings)

    outputs = model(
        image_embeddings=adapted_embeddings,
        input_boxes=inputs["input_boxes"],
        multimask_output=True,
    )
    best_idx = outputs.iou_scores[0, 0].argmax().item()
    logits = outputs.pred_masks[0, 0, best_idx]
    probability = torch.sigmoid(logits)

    height = inputs["original_sizes"][0, 0].item()
    width = inputs["original_sizes"][0, 1].item()
    probability = F.interpolate(
        probability[None, None], size=(height, width), mode="bilinear", align_corners=False
    )[0, 0]

    prediction = (probability > threshold).cpu().numpy().astype(np.uint8)
    confidence = outputs.iou_scores[0, 0, best_idx].float().item()
    return prediction, confidence, best_idx

print("SAM2 predictors ready.")

SAM2 predictors ready.


## 20. Evaluation helpers

In [64]:
def load_evaluation_item(record, dataset_name):
    if dataset_name == "VegAnn":
        sample = record["hf_dataset"][record["index"]]
        image = sample["image"]
        gt_mask = binary_mask_from_array(sample["mask"])
    elif dataset_name == "ReLeaf":
        image, gt_mask = load_releaf_image_mask(record["image_path"])
    elif dataset_name == "WeedsGalore":
        image, gt_mask = load_weedsgalore_image_mask(record)
    elif dataset_name == "EWI1S":
        image, gt_mask = load_ewis1_image_mask(record)
    else:
        raise ValueError(f"Unknown dataset: {dataset_name}")

    if not isinstance(image, Image.Image):
        image = Image.fromarray(np.asarray(image))
    return image.convert("RGB"), np.asarray(gt_mask).astype(bool)


evaluation_datasets = {
    "VegAnn": vegann_val_records,
    "ReLeaf": releaf_val_pairs,
    "WeedsGalore": weedsgalore_val_pairs,
    "EWI1S": ewis1_val_pairs,
}

print({k: len(v) for k, v in evaluation_datasets.items()})

{'VegAnn': 746, 'ReLeaf': 34, 'WeedsGalore': 15, 'EWI1S': 8}


In [65]:
def _metric_dict(prediction, gt_mask):
    prediction = prediction.astype(bool)
    if prediction.shape != gt_mask.shape:
        prediction = cv2.resize(
            prediction.astype(np.uint8),
            (gt_mask.shape[1], gt_mask.shape[0]),
            interpolation=cv2.INTER_NEAREST,
        ).astype(bool)

    intersection = np.logical_and(prediction, gt_mask).sum()
    union = np.logical_or(prediction, gt_mask).sum()
    pred_area = prediction.sum()
    gt_area = gt_mask.sum()

    iou = intersection / union if union > 0 else 0.0
    dice = (2 * intersection / (pred_area + gt_area)) if pred_area + gt_area > 0 else 0.0

    fp = np.logical_and(prediction, ~gt_mask).sum()
    fn = np.logical_and(~prediction, gt_mask).sum()
    precision = intersection / (intersection + fp) if intersection + fp > 0 else 0.0
    recall = intersection / (intersection + fn) if intersection + fn > 0 else 0.0

    return iou, dice, precision, recall

In [89]:
@torch.no_grad()
def evaluate_decoder_only(checkpoint_path, dataset_records, dataset_name):
    model, _ = create_sam2_from_decoder_checkpoint(checkpoint_path)
    model.eval()

    values = []
    for idx, record in enumerate(dataset_records):
        try:
            image, gt_mask = load_evaluation_item(record, dataset_name)
            if gt_mask.sum() == 0:
                continue
            box, _, _, _ = generate_agriclip_box(image, TOP_K, BOX_MARGIN)
            pred, conf, best_idx = sam2_predict_box_multimask(
                model, np.array(image), box, MASK_THRESHOLD
            )
            values.append((*_metric_dict(pred, gt_mask), conf, best_idx))
            if (idx + 1) % 25 == 0 or idx == 0:
                print(f"Decoder-only | {dataset_name} | {idx + 1}/{len(dataset_records)}")
        except Exception as e:
            print(f"ERROR decoder-only | {dataset_name} | image {idx}: {type(e).__name__}: {e}")

    result = {
        "checkpoint": "DecoderOnly",
        "dataset": dataset_name,
        "images_evaluated": len(values),
        "IoU_%": 100 * np.mean([x[0] for x in values]) if values else np.nan,
        "Dice_%": 100 * np.mean([x[1] for x in values]) if values else np.nan,
        "Precision_%": 100 * np.mean([x[2] for x in values]) if values else np.nan,
        "Recall_%": 100 * np.mean([x[3] for x in values]) if values else np.nan,
        "mean_confidence": np.mean([x[4] for x in values]) if values else np.nan,
    }
    del model
    torch.cuda.empty_cache()
    return result


@torch.no_grad()
# ============================================================
# FIXED: EVALUATE ENCODER ADAPTER
# ============================================================

# ============================================================
# FIXED ENCODER-ADAPTER EVALUATION
# ============================================================

@torch.no_grad()
def evaluate_adapter(adapter_checkpoint_path, dataset_records, dataset_name):

    checkpoint = torch.load(
        adapter_checkpoint_path,
        map_location="cpu",
        weights_only=False
    )

    # --------------------------------------------------------
    # Use THIS dataset's decoder checkpoint
    # --------------------------------------------------------
    decoder_checkpoint = DECODER_ONLY_CHECKPOINTS[dataset_name]

    model, _ = create_sam2_from_decoder_checkpoint(
        decoder_checkpoint
    )

    # --------------------------------------------------------
    # Restore the trained decoder
    # --------------------------------------------------------
    model.mask_decoder.load_state_dict(
        checkpoint["mask_decoder_state_dict"],
        strict=True
    )

    # --------------------------------------------------------
    # IMPORTANT:
    # Use MULTI-SCALE adapter
    # --------------------------------------------------------
    adapter = SAM2MultiScaleFeatureAdapter(
        channels=checkpoint["adapter_channels"],
        bottleneck=checkpoint["adapter_bottleneck"]
    ).to(DEVICE)

    adapter.load_state_dict(
        checkpoint["encoder_adapter_state_dict"],
        strict=True
    )

    model.eval()
    adapter.eval()

    print(
        f"Adapter checkpoint loaded for {dataset_name}"
    )

    print(
        "Adapter channels:",
        checkpoint["adapter_channels"]
    )

    print(
        "Adapter bottleneck:",
        checkpoint["adapter_bottleneck"]
    )

    values = []

    for idx, record in enumerate(dataset_records):

        try:

            # ------------------------------------------------
            # Load image + GT
            # ------------------------------------------------
            image, gt_mask = load_evaluation_item(
                record,
                dataset_name
            )

            if gt_mask.sum() == 0:
                continue

            # ------------------------------------------------
            # AgriCLIP box
            # ------------------------------------------------
            box, _, _, _ = generate_agriclip_box(
                image,
                TOP_K,
                BOX_MARGIN
            )

            # ------------------------------------------------
            # SAM2 + encoder adapter
            # ------------------------------------------------
            pred, conf, best_idx = (
                adapter_sam2_predict_box_multimask(
                    model,
                    adapter,
                    np.array(image),
                    box,
                    MASK_THRESHOLD
                )
            )

            values.append(
                (*_metric_dict(pred, gt_mask), conf, best_idx)
            )

            if (idx + 1) % 25 == 0 or idx == 0:

                print(
                    f"Adapter | {dataset_name} | "
                    f"{idx + 1}/{len(dataset_records)}"
                )

        except Exception as e:

            print(
                f"ERROR adapter | {dataset_name} | "
                f"image {idx}: "
                f"{type(e).__name__}: {e}"
            )

    # --------------------------------------------------------
    # Aggregate
    # --------------------------------------------------------
    result = {

        "checkpoint": "DecoderPlusAdapter",

        "dataset": dataset_name,

        "images_evaluated": len(values),

        "IoU_%":
            100 * np.mean([x[0] for x in values])
            if values else np.nan,

        "Dice_%":
            100 * np.mean([x[1] for x in values])
            if values else np.nan,

        "Precision_%":
            100 * np.mean([x[2] for x in values])
            if values else np.nan,

        "Recall_%":
            100 * np.mean([x[3] for x in values])
            if values else np.nan,

        "mean_confidence":
            np.mean([x[4] for x in values])
            if values else np.nan,
    }

    del model
    del adapter

    torch.cuda.empty_cache()

    return result

## 21. PRIMARY result — decoder-only vs. encoder adapter, all datasets
This is the headline comparison for the thesis: same AgriCLIP box, same evaluation protocol, decoder-only vs. decoder+adapter, on every dataset's own held-out split.

In [84]:
# ============================================================
# FIX: ROBUST DINO FEATURE EXTRACTION
# ============================================================

from PIL import Image
import numpy as np
import torch


def extract_dino_layer4_feature(image):
    """
    Extract DINO ResNet50 layer4 feature using the already-loaded
    dino_model. This avoids the failing hook implementation used
    elsewhere in the evaluation pipeline.
    
    Returns:
        Tensor of shape [1, 2048, 7, 7]
    """

    if not isinstance(image, Image.Image):
        image = Image.fromarray(np.array(image))

    x = dino_transform(image).unsqueeze(0).to(device)

    captured = {}

    def hook_fn(module, inputs, output):
        captured["feature"] = output

    handle = dino_model.layer4.register_forward_hook(hook_fn)

    try:
        with torch.no_grad():
            _ = dino_model(x)
    finally:
        handle.remove()

    if "feature" not in captured:
        raise RuntimeError(
            "DINO layer4 feature was not captured. "
            "The DINO forward pass did not trigger layer4."
        )

    feature = captured["feature"]

    if not torch.isfinite(feature).all():
        raise RuntimeError("DINO layer4 feature contains NaN/Inf values.")

    return feature

In [78]:
# ============================================================
# TEST FIXED DINO FEATURE EXTRACTION
# ============================================================

sample = evaluation_datasets["VegAnn"][0]

raw_sample = sample["hf_dataset"][sample["index"]]
test_image = raw_sample["image"]

test_feature = extract_dino_layer4_feature(test_image)

print("SUCCESS")
print("Feature shape:", tuple(test_feature.shape))
print("Finite:", torch.isfinite(test_feature).all().item())
print("Mean:", test_feature.mean().item())
print("Std:", test_feature.std().item())

SUCCESS: layer4 hook fired!
Output type: <class 'torch.Tensor'>
Output shape: (1, 2048, 7, 7)
SUCCESS
Feature shape: (1, 2048, 7, 7)
Finite: True
Mean: 0.04296372830867767
Std: 0.19958415627479553


In [80]:
# ============================================================
# FIND FUNCTIONS RELATED TO DINO / AGRICLIP / FEATURE / BOX
# ============================================================

import inspect

print("Functions currently available containing DINO/CLIP/feature/box/align:\n")

# Make a fixed list first so globals() cannot change during iteration
global_items = list(globals().items())

for name, obj in global_items:
    if callable(obj):
        name_lower = name.lower()

        if any(k in name_lower for k in [
            "dino",
            "clip",
            "feature",
            "box",
            "align"
        ]):
            print(name)

Functions currently available containing DINO/CLIP/feature/box/align:

mask_to_box
jitter_box
SAM2FeatureAdapter
SAM2MultiScaleFeatureAdapter
sam2_predict_box_multimask
adapter_sam2_predict_box_multimask
LinearAligner
dino_transform
dino_model
extract_dino_patch_features
align_patch_features
generate_agriclip_box
extract_dino_layer4_feature


In [85]:
inspect.getsource(generate_agriclip_box)

'@torch.no_grad()\ndef generate_agriclip_box(image, top_k=32, margin=25):\n    patch_features, grid_h, grid_w = extract_dino_patch_features(image)\n    aligned_features = align_patch_features(patch_features)\n    patch_scores = aligned_features @ vegetation_embedding\n    scores = patch_scores.cpu().numpy()\n    topk_indices = np.argsort(scores)[::-1][:top_k]\n\n    W, H = image.size\n    patch_h = H / grid_h\n    patch_w = W / grid_w\n\n    boxes = []\n    for idx in topk_indices:\n        r = idx // grid_w\n        c = idx % grid_w\n        boxes.append([\n            int(c * patch_w), int(r * patch_h),\n            int((c + 1) * patch_w), int((r + 1) * patch_h),\n        ])\n    boxes = np.asarray(boxes)\n\n    xmin = max(0, boxes[:, 0].min() - margin)\n    ymin = max(0, boxes[:, 1].min() - margin)\n    xmax = min(W - 1, boxes[:, 2].max() + margin)\n    ymax = min(H - 1, boxes[:, 3].max() + margin)\n\n    return np.array([xmin, ymin, xmax, ymax], dtype=np.int32), boxes, scores, topk

In [87]:
# ============================================================
# TEST FIXED DINO PATCH EXTRACTION
# ============================================================

sample = evaluation_datasets["VegAnn"][0]

raw_sample = sample["hf_dataset"][sample["index"]]

test_image = raw_sample["image"]

patch_features, grid_h, grid_w = extract_dino_patch_features(
    test_image
)

print("SUCCESS")
print()
print("Grid:", grid_h, "x", grid_w)
print("Number of patches:", patch_features.shape[0])
print("Feature dimension:", patch_features.shape[1])
print("Feature shape:", tuple(patch_features.shape))
print("Finite:", torch.isfinite(patch_features).all().item())

SUCCESS: layer4 hook fired!
Output type: <class 'torch.Tensor'>
Output shape: (1, 2048, 7, 7)
SUCCESS

Grid: 7 x 7
Number of patches: 49
Feature dimension: 2048
Feature shape: (49, 2048)
Finite: True


In [88]:
# ============================================================
# TEST COMPLETE AGRICLIP BOX GENERATION
# ============================================================

box, boxes, scores, topk_indices = generate_agriclip_box(
    test_image,
    TOP_K,
    BOX_MARGIN
)

print("SUCCESS")
print()
print("Final AgriCLIP box:", box)
print("All patch boxes shape:", boxes.shape)
print("Scores shape:", scores.shape)
print("Top-K indices shape:", topk_indices.shape)
print("TOP_K:", TOP_K)
print("Highest scoring patch:", topk_indices[0])
print("Lowest selected patch:", topk_indices[-1])

SUCCESS: layer4 hook fired!
Output type: <class 'torch.Tensor'>
Output shape: (1, 2048, 7, 7)
SUCCESS

Final AgriCLIP box: [  0   0 511 511]
All patch boxes shape: (32, 4)
Scores shape: (49,)
Top-K indices shape: (32,)
TOP_K: 32
Highest scoring patch: 5
Lowest selected patch: 14


In [90]:
# ============================================================
# ONE-IMAGE END-TO-END SANITY CHECK
# ============================================================

dataset_name = "VegAnn"

record = evaluation_datasets[dataset_name][0]

image, gt_mask = load_evaluation_item(
    record,
    dataset_name
)

print("Image size:", image.size)
print("GT mask shape:", gt_mask.shape)
print("GT foreground pixels:", gt_mask.sum())

# ------------------------------------------------------------
# Generate AgriCLIP box
# ------------------------------------------------------------

box, boxes, scores, topk_indices = generate_agriclip_box(
    image,
    TOP_K,
    BOX_MARGIN
)

print("\nAgriCLIP box:", box)
print("Selected patches:", len(topk_indices))

# ------------------------------------------------------------
# Decoder-only
# ------------------------------------------------------------

decoder_model, _ = create_sam2_from_decoder_checkpoint(
    DECODER_ONLY_CHECKPOINTS[dataset_name]
)

decoder_model.eval()

pred_decoder, conf_decoder, best_decoder = (
    sam2_predict_box_multimask(
        decoder_model,
        np.array(image),
        box,
        MASK_THRESHOLD
    )
)

decoder_metrics = _metric_dict(
    pred_decoder,
    gt_mask
)

print("\nDecoder-only:")
print("IoU:", decoder_metrics[0])
print("Dice:", decoder_metrics[1])
print("Precision:", decoder_metrics[2])
print("Recall:", decoder_metrics[3])
print("Confidence:", conf_decoder)

del decoder_model
torch.cuda.empty_cache()

[transformers] You are using a model of type `sam2_video` to instantiate a model of type `sam2`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Image size: (512, 512)
GT mask shape: (512, 512)
GT foreground pixels: 208628
SUCCESS: layer4 hook fired!
Output type: <class 'torch.Tensor'>
Output shape: (1, 2048, 7, 7)

AgriCLIP box: [  0   0 511 511]
Selected patches: 32


Loading weights:   0%|          | 0/309 [00:00<?, ?it/s]


Decoder-only:
IoU: 0.9021771316893613
Dice: 0.9485732076782143
Precision: 0.9479171154094689
Recall: 0.9492302087926836
Confidence: 0.9524438381195068


In [91]:
# ============================================================
# PHASE 2 PRIMARY COMPARISON
# DECODER-ONLY vs ENCODER-ADAPTER
# ============================================================

_missing_adapters = [
    n for n in ADAPTER_DATASETS
    if not ADAPTER_CHECKPOINTS[n].exists()
]

if _missing_adapters:
    raise FileNotFoundError(
        f"Missing adapter checkpoint(s) for: {_missing_adapters}. "
        "Run section 16 first."
    )

phase2_primary_results = []

for dataset_name in ADAPTER_DATASETS:

    print("\n" + "=" * 70)
    print(f"DATASET: {dataset_name}")
    print("=" * 70)

    # --------------------------------------------------------
    # DECODER ONLY
    # --------------------------------------------------------
    print(f"\nEvaluating decoder-only: {dataset_name}")

    decoder_result = evaluate_decoder_only(
        DECODER_ONLY_CHECKPOINTS[dataset_name],
        evaluation_datasets[dataset_name],
        dataset_name
    )

    phase2_primary_results.append(decoder_result)

    # --------------------------------------------------------
    # ENCODER ADAPTER
    # --------------------------------------------------------
    print(f"\nEvaluating encoder adapter: {dataset_name}")

    adapter_result = evaluate_adapter(
        ADAPTER_CHECKPOINTS[dataset_name],
        evaluation_datasets[dataset_name],
        dataset_name
    )

    phase2_primary_results.append(adapter_result)


# ============================================================
# RESULTS TABLE
# ============================================================

phase2_primary_df = pd.DataFrame(phase2_primary_results)

display(
    phase2_primary_df.round(3)
)


# ============================================================
# SAVE RESULTS
# ============================================================

output_csv = (
    DOMAIN_FT_ROOT /
    "PRIMARY_AGRICLIP_COMPARISON_ALL_DATASETS.csv"
)

phase2_primary_df.to_csv(
    output_csv,
    index=False
)

print(f"\nSaved results to:\n{output_csv}")


# ============================================================
# IOU COMPARISON
# ============================================================

primary_pivot = phase2_primary_df.pivot(
    index="dataset",
    columns="checkpoint",
    values="IoU_%"
)

print("\nIoU_% by dataset and method:")

display(
    primary_pivot.round(2)
)

[transformers] You are using a model of type `sam2_video` to instantiate a model of type `sam2`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.



DATASET: VegAnn

Evaluating decoder-only: VegAnn


Loading weights:   0%|          | 0/309 [00:00<?, ?it/s]

SUCCESS: layer4 hook fired!
Output type: <class 'torch.Tensor'>
Output shape: (1, 2048, 7, 7)
Decoder-only | VegAnn | 1/746
SUCCESS: layer4 hook fired!
Output type: <class 'torch.Tensor'>
Output shape: (1, 2048, 7, 7)
SUCCESS: layer4 hook fired!
Output type: <class 'torch.Tensor'>
Output shape: (1, 2048, 7, 7)
SUCCESS: layer4 hook fired!
Output type: <class 'torch.Tensor'>
Output shape: (1, 2048, 7, 7)
SUCCESS: layer4 hook fired!
Output type: <class 'torch.Tensor'>
Output shape: (1, 2048, 7, 7)
SUCCESS: layer4 hook fired!
Output type: <class 'torch.Tensor'>
Output shape: (1, 2048, 7, 7)
SUCCESS: layer4 hook fired!
Output type: <class 'torch.Tensor'>
Output shape: (1, 2048, 7, 7)
SUCCESS: layer4 hook fired!
Output type: <class 'torch.Tensor'>
Output shape: (1, 2048, 7, 7)
SUCCESS: layer4 hook fired!
Output type: <class 'torch.Tensor'>
Output shape: (1, 2048, 7, 7)
SUCCESS: layer4 hook fired!
Output type: <class 'torch.Tensor'>
Output shape: (1, 2048, 7, 7)
SUCCESS: layer4 hook fired!
Ou

[transformers] You are using a model of type `sam2_video` to instantiate a model of type `sam2`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Loading weights:   0%|          | 0/309 [00:00<?, ?it/s]

... (19 outputs left out)

Adapter checkpoint loaded for EWI1S
Adapter channels: (32, 64, 256)
Adapter bottleneck: 32
SUCCESS: layer4 hook fired!
Output type: <class 'torch.Tensor'>
Output shape: (1, 2048, 7, 7)
Adapter | EWI1S | 1/8
SUCCESS: layer4 hook fired!
Output type: <class 'torch.Tensor'>
Output shape: (1, 2048, 7, 7)
SUCCESS: layer4 hook fired!
Output type: <class 'torch.Tensor'>
Output shape: (1, 2048, 7, 7)
SUCCESS: layer4 hook fired!
Output type: <class 'torch.Tensor'>
Output shape: (1, 2048, 7, 7)
SUCCESS: layer4 hook fired!
Output type: <class 'torch.Tensor'>
Output shape: (1, 2048, 7, 7)
SUCCESS: layer4 hook fired!
Output type: <class 'torch.Tensor'>
Output shape: (1, 2048, 7, 7)
SUCCESS: layer4 hook fired!
Output type: <class 'torch.Tensor'>
Output shape: (1, 2048, 7, 7)
SUCCESS: layer4 hook fired!
Output type: <class 'torch.Tensor'>
Output shape: (1, 2048, 7, 7)


,checkpoint,dataset,images_evaluated,IoU_%,Dice_%,Precision_%,Recall_%,mean_confidence
0,DecoderOnly,VegAnn,746,85.058,91.178,90.094,93.417,0.942
1,DecoderPlusAdapter,VegAnn,746,85.437,91.472,91.160,92.693,0.943
2,DecoderOnly,ReLeaf,34,84.695,91.439,87.178,96.524,0.969
3,DecoderPlusAdapter,ReLeaf,34,85.473,91.927,88.132,96.371,0.969
4,DecoderOnly,WeedsGalore,15,62.195,76.350,76.368,77.056,0.900
5,DecoderPlusAdapter,WeedsGalore,15,62.111,76.275,75.099,78.246,0.904
6,DecoderOnly,EWI1S,8,17.712,26.039,30.231,27.200,0.772
7,DecoderPlusAdapter,EWI1S,8,17.754,26.056,32.384,25.883,0.766



Saved results to:
/home/zahranadeem/thesis/sam2_domain_finetuning/PRIMARY_AGRICLIP_COMPARISON_ALL_DATASETS.csv

IoU_% by dataset and method:


checkpoint,DecoderOnly,DecoderPlusAdapter
dataset,,
EWI1S,17.71,17.75
ReLeaf,84.69,85.47
VegAnn,85.06,85.44
WeedsGalore,62.19,62.11


## 22. Optional — full cross-domain grid
4 checkpoints × 2 methods × 4 eval sets = 32 full AgriCLIP+SAM2 evaluation passes. This is expensive — only run it if you want the full generalization picture beyond same-domain performance.

In [92]:
phase2_cross_domain = []

for train_dataset_name in ADAPTER_DATASETS:
    for eval_dataset_name in ADAPTER_DATASETS:
        row_decoder = evaluate_decoder_only(
            DECODER_ONLY_CHECKPOINTS[train_dataset_name],
            evaluation_datasets[eval_dataset_name],
            eval_dataset_name,
        )
        row_decoder["trained_on"] = train_dataset_name
        phase2_cross_domain.append(row_decoder)

        row_adapter = evaluate_adapter(
            ADAPTER_CHECKPOINTS[train_dataset_name],
            evaluation_datasets[eval_dataset_name],
            eval_dataset_name,
        )
        row_adapter["trained_on"] = train_dataset_name
        phase2_cross_domain.append(row_adapter)

phase2_cross_domain_df = pd.DataFrame(phase2_cross_domain)
display(phase2_cross_domain_df.round(3))
phase2_cross_domain_df.to_csv(DOMAIN_FT_ROOT / "CROSS_DOMAIN_AGRICLIP_COMPARISON_ALL.csv", index=False)

[transformers] You are using a model of type `sam2_video` to instantiate a model of type `sam2`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Loading weights:   0%|          | 0/309 [00:00<?, ?it/s]

SUCCESS: layer4 hook fired!
Output type: <class 'torch.Tensor'>
Output shape: (1, 2048, 7, 7)
Decoder-only | VegAnn | 1/746
SUCCESS: layer4 hook fired!
Output type: <class 'torch.Tensor'>
Output shape: (1, 2048, 7, 7)
SUCCESS: layer4 hook fired!
Output type: <class 'torch.Tensor'>
Output shape: (1, 2048, 7, 7)
SUCCESS: layer4 hook fired!
Output type: <class 'torch.Tensor'>
Output shape: (1, 2048, 7, 7)
SUCCESS: layer4 hook fired!
Output type: <class 'torch.Tensor'>
Output shape: (1, 2048, 7, 7)
SUCCESS: layer4 hook fired!
Output type: <class 'torch.Tensor'>
Output shape: (1, 2048, 7, 7)
SUCCESS: layer4 hook fired!
Output type: <class 'torch.Tensor'>
Output shape: (1, 2048, 7, 7)
SUCCESS: layer4 hook fired!
Output type: <class 'torch.Tensor'>
Output shape: (1, 2048, 7, 7)
SUCCESS: layer4 hook fired!
Output type: <class 'torch.Tensor'>
Output shape: (1, 2048, 7, 7)
SUCCESS: layer4 hook fired!
Output type: <class 'torch.Tensor'>
Output shape: (1, 2048, 7, 7)
SUCCESS: layer4 hook fired!
Ou

[transformers] You are using a model of type `sam2_video` to instantiate a model of type `sam2`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Loading weights:   0%|          | 0/309 [00:00<?, ?it/s]

Adapter checkpoint loaded for VegAnn
Adapter channels: (32, 64, 256)
Adapter bottleneck: 32
SUCCESS: layer4 hook fired!
Output type: <class 'torch.Tensor'>
Output shape: (1, 2048, 7, 7)
Adapter | VegAnn | 1/746
SUCCESS: layer4 hook fired!
Output type: <class 'torch.Tensor'>
Output shape: (1, 2048, 7, 7)
SUCCESS: layer4 hook fired!
Output type: <class 'torch.Tensor'>
Output shape: (1, 2048, 7, 7)
SUCCESS: layer4 hook fired!
Output type: <class 'torch.Tensor'>
Output shape: (1, 2048, 7, 7)
SUCCESS: layer4 hook fired!
Output type: <class 'torch.Tensor'>
Output shape: (1, 2048, 7, 7)
SUCCESS: layer4 hook fired!
Output type: <class 'torch.Tensor'>
Output shape: (1, 2048, 7, 7)
SUCCESS: layer4 hook fired!
Output type: <class 'torch.Tensor'>
Output shape: (1, 2048, 7, 7)
SUCCESS: layer4 hook fired!
Output type: <class 'torch.Tensor'>
Output shape: (1, 2048, 7, 7)
SUCCESS: layer4 hook fired!
Output type: <class 'torch.Tensor'>
Output shape: (1, 2048, 7, 7)
SUCCESS: layer4 hook fired!
Output ty

... (90 outputs left out)

[transformers] You are using a model of type `sam2_video` to instantiate a model of type `sam2`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Loading weights:   0%|          | 0/309 [00:00<?, ?it/s]

Adapter checkpoint loaded for EWI1S
Adapter channels: (32, 64, 256)
Adapter bottleneck: 32
SUCCESS: layer4 hook fired!
Output type: <class 'torch.Tensor'>
Output shape: (1, 2048, 7, 7)
Adapter | EWI1S | 1/8
SUCCESS: layer4 hook fired!
Output type: <class 'torch.Tensor'>
Output shape: (1, 2048, 7, 7)
SUCCESS: layer4 hook fired!
Output type: <class 'torch.Tensor'>
Output shape: (1, 2048, 7, 7)
SUCCESS: layer4 hook fired!
Output type: <class 'torch.Tensor'>
Output shape: (1, 2048, 7, 7)
SUCCESS: layer4 hook fired!
Output type: <class 'torch.Tensor'>
Output shape: (1, 2048, 7, 7)
SUCCESS: layer4 hook fired!
Output type: <class 'torch.Tensor'>
Output shape: (1, 2048, 7, 7)
SUCCESS: layer4 hook fired!
Output type: <class 'torch.Tensor'>
Output shape: (1, 2048, 7, 7)
SUCCESS: layer4 hook fired!
Output type: <class 'torch.Tensor'>
Output shape: (1, 2048, 7, 7)


,checkpoint,dataset,images_evaluated,IoU_%,Dice_%,Precision_%,Recall_%,mean_confidence,trained_on
0,DecoderOnly,VegAnn,746,85.058,91.178,90.094,93.417,0.942,VegAnn
1,DecoderPlusAdapter,VegAnn,746,85.437,91.472,91.160,92.693,0.943,VegAnn
2,DecoderOnly,ReLeaf,34,79.107,87.802,80.750,97.393,0.962,VegAnn
3,DecoderPlusAdapter,ReLeaf,34,79.601,88.119,81.532,97.008,0.963,VegAnn
4,DecoderOnly,WeedsGalore,15,55.814,71.076,73.951,69.507,0.864,VegAnn
5,DecoderPlusAdapter,WeedsGalore,15,53.593,69.227,77.389,63.714,0.859,VegAnn
6,DecoderOnly,EWI1S,8,12.184,17.291,13.654,64.441,0.824,VegAnn
7,DecoderPlusAdapter,EWI1S,8,12.187,17.161,14.210,61.284,0.814,VegAnn
8,DecoderOnly,VegAnn,746,76.016,85.219,93.393,80.145,0.917,ReLeaf
9,DecoderPlusAdapter,VegAnn,746,72.547,82.712,93.435,76.303,0.912,ReLeaf
